# 04r Amendment 2 to the coding rules, and every record re-coded under it

Design log.
- Why this notebook exists. The second pass (notebook 04_second_pass) put seven fields below the plan v2 A6 threshold: defence isolation, signal dependence, axis 1, and coverage of R1, R3ns, R3cp and R3safe. Plan A6 says such fields are re-coded in full. The disagreements showed three gaps in the rules, so the rules are amended first and every record is then re-coded under the amended text.
- What the gaps were, in plain words:
  - Isolation (rule 8). The rule said when to write `ablation_present` but never said what `not_applicable` means. The first coding used it for variants and for other papers' defences; the second coding used `no_ablation` or `ablation_present` for the same records.
  - Signal dependence (rule 9). `baseline_only` was defined only for prompting defences, and `not_applicable` ("no signal at inference") was sometimes given to defences that do show a signal.
  - Axis 1 (rule 7). "Chat roles" had no test. A defence trained inside an agent harness that already has a tool role was coded `chat_roles` by one coding and `no_signal` by the other.
  - Coverage (rule 32). Pooled results, answer flips and metrics that need instruction-bearing data were coded in different ways.
- Amendment 2, short form (full text in `schema/coding_rules.md`, rules 7, 8, 9 and 32):
  - Axis 1: `chat_roles` only when the method itself adds a role or turn for the untrusted text, trains an order of privilege over roles, or reads each segment's role in its inference rule. Roles the template or harness already has do not count alone. Prompt wording and number-only labels are `no_signal`. An internal intervention counts only when it acts by source.
  - Isolation: coded for the method. `no_ablation` for a method with no ablation run; `not_applicable` for a variant or control record, and for another paper's defence that this report does not ablate.
  - Signal dependence: `not_applicable` exactly when axis 1 is `no_signal`. A prompting defence or fixed rule with a signal is `measured` if run without the signal, else `baseline_only` if the undefended run is reported, else `not_reported`. A trained defence is never `baseline_only`.
  - Coverage: `aggregate_only` whenever a quantity is reported only pooled. R1 counts any injected instruction executed; an answer flip is an R1 proxy. Task done and attack failed together is R3safe. A score on needed data that carries instruction-bearing text is an R3cp proxy.
- Order of work. The agreement figures of 04_second_pass must be committed before any record changes, so Stage 1 stops unless they are. The re-code is a list of changed fields (`reports/recode_04r_changes.csv`), each with its old value, new value, reason and page. Stage 3 refuses any row whose old value is not the value on file, checks every record file before and after against fixed SHA-256 digests, and checks the amended rules on every record.
- What changed: 525 fields in 74 of 124 record files; axis 1 in 27 defence records, isolation in 138, signal dependence in 294; coverage, result and role fields as listed in the change file. Printed values, anchors and comparisons are not touched.
- Fields that met the threshold are not re-coded in full, but each of their second-pass disagreements is resolved one by one in `reports/second_pass_resolution_04r.csv`, with the value kept and the reason.
- Outputs: `schema/coding_rules.md` (Amendment 2), `src/extract.py` and `tests/test_extract.py` (the change applier and the amended rules as checks), `reports/records/` (re-coded), `reports/recode_04r_changes.csv`, `reports/recode_04r.csv` (every field re-checked), `reports/second_pass_resolution_04r.csv`, `reports/decision_04r.txt`.
- CPU only; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for mod, pkg in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 1. Gate: the second-pass agreement figures are committed before any record changes

In [ ]:
NEED = ["reports/agreement_04.csv", "reports/decision_04sp.txt", "reports/second_pass_disagreements_04.csv",
        "reports/second_pass_ids.txt", "src/extract.py"]
DIS_SHA = "9fb3d100c80725fc67bc6c47d1c30517b3d20cefde5f2a459fb3240bc75eb080"          # the disagreement file this re-code was built against
import hashlib
missing = [p for p in NEED if not Path(p).exists()]
untracked = subprocess.run(["git", "ls-files", "--error-unmatch", *[p for p in NEED if Path(p).exists()]],
                           capture_output=True, text=True).returncode != 0
dirty = subprocess.run(["git", "status", "--porcelain", "--", *NEED], capture_output=True, text=True).stdout.strip()
if missing or untracked or dirty:
    raise SystemExit("STOP: run and commit notebook 04_second_pass first; its agreement figures must be in the repository "
                     f"before any record changes (missing: {missing}; not committed or changed: {bool(untracked) or dirty})")
got = hashlib.sha256(Path("reports/second_pass_disagreements_04.csv").read_bytes()).hexdigest()
if got != DIS_SHA:
    raise SystemExit("STOP: reports/second_pass_disagreements_04.csv differs from the file this re-code was built against")
SP_COMMIT = subprocess.run(["git", "log", "-1", "--format=%h %s", "--", "reports/agreement_04.csv"], capture_output=True, text=True).stdout.strip()
print("agreement figures committed in:", SP_COMMIT)

## Stage 2. Amendment 2 to the coding rules; the change applier and the amended rules as code

In [ ]:
%%writefile schema/coding_rules.md
# Coding rules for the record files (frozen with schema v1 in notebook 03)

These rules say how a paper is turned into a record file in `reports/records/<report_id>.yaml`. They were committed before full extraction (notebook 04). Amendment 2 (notebook 04r) was added after the second pass of notebook 04_second_pass showed gaps in rules 7, 8, 9 and 32; the text before each amendment is unchanged, and every record was re-coded under it. The schema in `schema/record_schema.yaml` says which fields exist and which values they take; these rules say how to choose the values.

## 1. Reading and pages

1. Every record is filled from the PDF in `data/defences/`, whose SHA-256 is in `reports/defences.csv`. Nothing is filled from memory, from an abstract alone or from another paper's description of the work.
2. Every record holding a fact from the paper carries `page`, the 1-based page index of the PDF (not the printed page number). A fact on a different page goes in `pages: {field: page}`.
3. `extraction_scope` says what was coded: "whole report", or the sections and table rows coded. A quantity outside the scope is never coded "not reported".
4. `read: F` means the full text was read. Main-table rows need F.

## 2. Defences

5. One defence record per method or variant on one base checkpoint. A method from another paper that this report runs is a defence record with `origin: other_report`.
6. `name` is the method's own name. When the paper prints one name but runs another checkpoint (SecFid prints "SecAlign" and runs facebook/Meta-SecAlign-8B), `name_as_printed` keeps the printed name and `checkpoint_id` names what was run.
7. Axis 1 (how the source is shown at inference) and axis 2 (how the behaviour was built) take every value that applies. A fixed, non-learned change to the data tokens' embeddings (ASIDE) is `fixed_embedding_transform`, not a learned token feature. Amendment 2 (axis 1): `chat_roles` only when the method itself adds a role or turn for the untrusted text, trains an order of privilege over roles, or uses each segment's role as an input to its inference rule; roles that the chat template or the agent harness already has do not count on their own. Wording in a prompt alone, and labels that are only numbers (numbered references), are `no_signal`. `internal_intervention` only when the intervention acts by source (on the tokens marked as untrusted, or keyed by their source); an intervention applied at a fixed position or by another score, wherever the untrusted text sits, is `no_signal`. A defence from another report whose layout this report does not describe takes axis 1 from its own report, or from how the other re-runs of the method in the corpus are coded.
8. `isolation: ablation_present` only when the paper runs the defence with a component removed or changed and reports the result. Amendment 2: isolation belongs to the method, so every base-checkpoint record of the method takes the same value. `no_ablation` is a method the report runs as its own with no such run. `not_applicable` is a record that is itself a variant or a control of another record in the same report (an ablation arm; an objective, data or training-stage variant; a checkpoint other than the final one; a control direction or prompt; the comparison arm of a training method), or another report's defence that this report runs without removing or changing its components.
9. Signal dependence: `measured` (the defence run with its signal removed or moved, with numbers); `measured_other_checkpoint` (measured, but on another checkpoint of the method, named in `signal_dependence_detail`); `stated` (the paper says the signal is needed, without numbers); `baseline_only` (a prompting defence whose signal-removed condition is just the instruction-only baseline); `not_reported`; `not_applicable` (no inference-time signal). Amendment 2: `not_applicable` exactly when axis 1 is `no_signal`. A prompting defence or a fixed inference rule with a signal is `measured` when the same defence is run with the signal removed or moved, otherwise `baseline_only` when the run without the defence is reported (the undefended model; for a prompt stacked on another defence, that defence alone), otherwise `not_reported`; `stated` is not used for these. A trained defence with a signal is `measured`, `measured_other_checkpoint`, `stated` or `not_reported`, never `baseline_only`.
10. Layout compatibility marks only layouts the paper ran (`supported` or `unsupported`); everything else is `untested`.
11. Amendment 1 (training targets): `trains_instruction_pair` is yes when training pairs the user's own instruction with the same injected words, partly when some injected strings also appear as legitimate instructions with other data, no otherwise. `trains_carry_through` is yes_exact_targets or yes_by_preference when training rewards carrying instruction-bearing text through as data. Both are `not_applicable` when nothing is trained: prompting defences and fixed inference rules, including a steering direction fitted from examples.

## 3. Evaluations

12. One evaluation per configuration under one protocol: one defence (or undefended model), one benchmark split, one attacker, one judge. An injection position reported separately (start, middle, end) is a separate evaluation.
13. The undefended reference is `defence_id: none:<base>`, with `baseline_kind`: off_the_shelf; same_data_twin (instruction-tuned by the authors on the same data, as ASIDE's Vanilla); instruction_prompt_only.
14. `measured_by` is the report whose authors ran the evaluation. A re-measurement of a released checkpoint carries `same_checkpoint_as: <report_id>:<defence_id>` only when the text says the released checkpoint was used.
15. Attacker: static (fixed injections), optimisation (gradient or search on this model), adaptive (built against this defence), human, rl. Access and budget as stated; "unreported" when not given.
16. Decoding as stated. When both arms of a within-report comparison ran in one harness and the paper does not print the setting, the condition is `matched_unstated`.
17. `run` names the run when a paper reports more than one run of the same configuration (canonical, comparison battery, replicate).
18. `subset_of` names the parent set when an evaluation uses a subset; `locked_split` says whether the test split was fixed before testing.
19. `judge_reading` is strict or relaxed when the paper defines both; otherwise single, or absent when not stated.

## 4. Results and the outcome coding

20. One result per printed number, with the metric name as printed and the value as printed. `value_source` says where it is printed. A result shown only in a chart is `value_source: figure_only` with no value; it counts for coverage and is never used in a comparison or a derived number.
21. `qualifier`: exact; approximate ("about 50%"); upper_bound ("below 3%"); lower_bound (CounterSteer's compromise rates under a generation budget).
22. A value relative to a reference uses `unit: percent_of_reference` and names the reference in `reference` (SEP utility set to 50.0 for the undefended model; benign utility as a percentage of the same row's clean run).
23. `spread` keeps the printed spread and the number of runs. Printed differences between two results are not coded; they are recomputed from the results.
24. Quantities: R1 from attack success, compromise or execution rates (R1 = 1 − rate). R2clean is the user's task done on clean data; R2attack under attack. R3ns is non-suppression (SecFid's fidelity, 1 − ignored rate); R3cp is correct handling of the data the task asked for (SecFid's processed rate); R3safe is processed and not executed.
25. AlpacaEval, MMLU and similar general benchmarks are `general_utility`, never R2. SEP utility and "utility under attack" scored on the user's task are R2 proxies unless the paper scores the user's instruction directly. AgentDojo's task checker scores the user's task and is R2 direct. A summary-quality score is at most an R3 proxy. The SEP separation score maps to no single quantity and is `other`.
26. When one printed metric is two quantities at once (SecFid's processed rate in translation and editing is R3cp and an R2attack proxy), the first goes in `quantity` and the second in `also_quantity`.
27. A string-matched attack success that also fires on quoted text is E or Rp in the outcome coding; it is coded Rp when the task asked for the text.

## 5. Comparisons and grades

28. A comparison has a purpose: W (which whole configuration does better), M (does this component cause the change) or X (a reading across two reports). `varies` says what differs between the arms.
29. Six conditions: c1 base checkpoint, c2 layout, c3 benchmark and split, c4 attacker, c5 judge and denominator, c6 decoding. Each is matched, matched_unstated, unmatched, unreported or manipulated (the one condition an M comparison varies on purpose).
30. Grade (plan v2 equation 4), computed by code: A when every condition is matched or is the one manipulated condition; B when exactly one is unmatched; C when two or more are; U when any is unreported. In W the layout may differ by design and is left out. matched_unstated counts as matched within one report and as unreported in an X comparison.

## 6. Claims and coverage

31. A claim is a headline claim from the abstract or introduction, with a verbatim anchor of at most 14 words, the jobs it speaks to, the comparisons that support it and, in `contradicted_by`, any result in the same report that cuts against it.
32. Coverage per quantity: direct; proxy (instrument named); aggregate_only (reported only pooled over defences, models or tasks); not_reported (inside the extraction scope); unclear; not_applicable. `where` points to the table or section. Amendment 2: `aggregate_only` applies whenever the quantity is reported only pooled, whatever the instrument, and `where` names the instrument. R1 counts the execution of any injected instruction, malicious or not; a changed answer counted whatever the injected goal (an answer flip) is an R1 proxy. A metric that scores the task done and the attack failed together is R3safe. A score on data the task needs that itself carries instruction-bearing text (a cue the agent must act on, valid evidence with an injection inside) is an R3cp proxy, in `also_quantity` when the metric is first another quantity.
33. A task partition that changes how a result should be read (CounterSteer's delegated-authority tasks) is described in a residual note; the schema has no field for it.

## 7. What is checked by code

34. Allowed values, required fields, references between records, page within the PDF, every printed value found on its page (in the plain text or in a column-preserving extraction, so table cells are not glued together), every anchor found on its page (a page break is allowed), anchors at most 14 words, and the PDF's SHA-256 against the manifest.

In [ ]:
%%writefile src/extract.py
"""Record files to CSV tables, the second-pass sample, value cards and agreement statistics (plan v2, B4).

Notebook 04 writes this module. Nothing here calls a model or the network."""
from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter, defaultdict

SEED = 2026
SHARE = 0.2
CARDS_PER_REPORT = 6
QUANTITIES = ["R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe"]
DEF_CATEGORICAL = ["origin", "isolation", "signal_dependence", "trains_instruction_pair", "trains_carry_through"]
DEF_MULTI = ["axis1", "axis2"]
EXACT_MIN, KAPPA_MIN, JACCARD_MIN = 0.8, 0.6, 0.8
MISSING = "(missing)"


def rank_key(seed, *parts) -> str:
    return hashlib.sha256("|".join([str(seed), *map(str, parts)]).encode()).hexdigest()


def second_pass_sample(included_slugs, seed: int = SEED, share: float = SHARE) -> list[str]:
    """The seeded sample: every included study ranked by a hash of the seed and its slug, the first
    ceil(share * N) kept. Removing a study from the frame does not reorder the others."""
    slugs = sorted(set(included_slugs))
    n = math.ceil(share * len(slugs))
    return sorted(slugs, key=lambda s: rank_key(seed, s))[:n]


def value_cards(rec: dict, k: int = CARDS_PER_REPORT, seed: int = SEED) -> list[dict]:
    """Up to k printed results per report, chosen by hash rank, given by their address only (never the value)."""
    rid = rec["report"]["id"]
    evs = {e["id"]: e for e in rec.get("evaluations") or []}
    dfs = {d["id"]: d for d in rec.get("defences") or []}
    printed = [r for r in rec.get("results") or [] if r.get("value_source") != "figure_only" and r.get("value") not in (None, "")]
    printed.sort(key=lambda r: rank_key(seed, rid, r["id"]))
    cards = []
    for r in printed[:k]:
        e = evs[r["evaluation_id"]]
        d = dfs.get(e["defence_id"])
        arm = f"{d['name']} on {d['base_checkpoint']}" if d else f"no defence ({str(e['defence_id'])[5:]}, {e.get('baseline_kind', '')})"
        extra = "; ".join(f"{f}: {e[f]}" for f in ("split", "run", "injection_position", "attacker_budget", "subset_of") if e.get(f))
        cards.append({"report": rid, "result": r["id"], "page": r.get("page"), "location": r.get("location", ""),
                      "arm": arm, "benchmark": e["benchmark"], "attacker": e["attacker_type"], "judge": e["judge"],
                      "metric": r["metric_as_printed"], "quantity": r["quantity"], "unit": r["unit"], "detail": extra})
    return cards


def _cell(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return ";".join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False)
    return v


def build_tables(records: dict, schema: dict, grade) -> dict:
    """The five extraction tables (defences, evaluations, results, comparisons, claims) and a report index, with
    ids made global as <report_id>:<local_id>. Derived columns: the defence each result belongs to, whether an
    evaluation is the original or a later measurement, and the grade of each comparison."""
    k = schema["kinds"]
    out = {name: [] for name in ("reports", "defences", "evaluations", "results", "comparisons", "claims")}
    for rid in sorted(records):
        rec = records[rid]
        rep, cov = rec["report"], rec.get("coverage") or {}
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        out["reports"].append({**{f: _cell(rep.get(f)) for f in k["report"]["fields"]},
                               **{f"cov_{q}": cov.get(q, "") for q in QUANTITIES},
                               "cov_where": _cell(cov.get("where")),
                               **{f"n_{x}": len(rec.get(x) or []) for x in ("defences", "evaluations", "results", "comparisons", "claims")}})
        for d in rec.get("defences") or []:
            out["defences"].append({"report_id": rid, "defence_uid": f"{rid}:{d['id']}", **{f: _cell(d.get(f)) for f in k["defences"]["fields"]}})

        def duid(did):
            did = str(did)
            return did if did.startswith("none:") or ":" in did else f"{rid}:{did}"

        def measurement(e):
            d = dfs.get(e["defence_id"])
            if d is None:
                return "undefended" if str(e["defence_id"]).startswith("none:") else "other_record"
            if d.get("origin") == "this_report" and e.get("measured_by") == rid:
                return "original"
            if d.get("origin") == "this_report":
                return "original_reprinted"
            return "later"

        for e in rec.get("evaluations") or []:
            out["evaluations"].append({"report_id": rid, "evaluation_uid": f"{rid}:{e['id']}", "defence_uid": duid(e["defence_id"]),
                                       "measurement": measurement(e), **{f: _cell(e.get(f)) for f in k["evaluations"]["fields"]}})
        for r in rec.get("results") or []:
            e = evs.get(r["evaluation_id"], {})
            out["results"].append({"report_id": rid, "result_uid": f"{rid}:{r['id']}", "evaluation_uid": f"{rid}:{r['evaluation_id']}",
                                   "defence_uid": duid(e.get("defence_id", "")), "measurement": measurement(e) if e else "",
                                   **{f: _cell(r.get(f)) for f in k["results"]["fields"]}})
        for c in rec.get("comparisons") or []:
            out["comparisons"].append({"report_id": rid, "comparison_uid": f"{rid}:{c['id']}", "left_uid": f"{rid}:{c['left']}",
                                       "right_uid": f"{rid}:{c['right']}", "grade": grade(c),
                                       **{f: _cell(c.get(f)) for f in k["comparisons"]["fields"]}})
        for cl in rec.get("claims") or []:
            row = {"report_id": rid, "claim_uid": f"{rid}:{cl['id']}", **{f: _cell(cl.get(f)) for f in k["claims"]["fields"]}}
            row["supported_by"] = ";".join(f"{rid}:{s}" for s in cl.get("supported_by") or [])
            row["contradicted_by"] = ";".join(f"{rid}:{s}" for s in cl.get("contradicted_by") or [])
            out["claims"].append(row)
    fields = {name: (list(rows[0].keys()) if rows else []) for name, rows in out.items()}
    return {name: (rows, fields[name]) for name, rows in out.items()}


def cross_record_problems(records: dict, included_slugs, page_counts: dict | None = None) -> list[str]:
    """Checks that need the whole corpus: one record per included study, references to other reports resolve,
    every record read in full, and (when page counts are given) every page within its PDF."""
    probs = []
    by_slug = defaultdict(list)
    for rid, rec in records.items():
        by_slug[rec["report"]["slug"]].append(rid)
    inc = set(included_slugs)
    for s in sorted(inc - set(by_slug)):
        probs.append(f"included study {s} has no record")
    for s in sorted(set(by_slug) - inc):
        probs.append(f"record(s) {by_slug[s]} belong to {s}, which is not an included study")
    for s, rids in by_slug.items():
        if len(rids) > 1:
            probs.append(f"included study {s} has {len(rids)} records: {rids}")
    defences = {f"{rid}:{d['id']}" for rid, rec in records.items() for d in rec.get("defences") or []}
    for rid, rec in records.items():
        if rec["report"].get("read") != "F":
            probs.append(f"{rid}: read is {rec['report'].get('read')!r}, not F")
        for e in rec.get("evaluations") or []:
            mb = e.get("measured_by")
            if mb != rid and mb not in records:
                probs.append(f"{rid}:{e['id']}: measured_by {mb!r} is neither this report nor another record")
            sc = e.get("same_checkpoint_as")
            if sc and sc not in defences:
                probs.append(f"{rid}:{e['id']}: same_checkpoint_as {sc!r} is not a defence in the corpus")
            did = str(e.get("defence_id", ""))
            if ":" in did and not did.startswith("none:") and did not in defences:
                probs.append(f"{rid}:{e['id']}: defence_id {did!r} is not a defence in the corpus")
        if page_counts and rid in page_counts:
            n = page_counts[rid]
            for kind in ("defences", "evaluations", "results", "comparisons", "claims"):
                for row in rec.get(kind) or []:
                    pages = [row.get("page")] + list((row.get("pages") or {}).values())
                    for p in pages:
                        if isinstance(p, int) and p > n:
                            probs.append(f"{rid}:{kind}:{row.get('id')}: page {p} beyond the PDF's {n} pages")
    return probs


def validate_second_pass(doc: dict, schema: dict, cards: list[dict], first: dict, match_line: list) -> list[str]:
    """Allowed values in a second-pass document, value answers for exactly the drawn cards, and match lines that
    point at first-pass defences of the same report."""
    k, rid, out = schema["kinds"], doc.get("report"), []
    if doc.get("role") not in k["report"]["fields"]["role"]["values"]:
        out.append(f"{rid}: role {doc.get('role')!r} not allowed")
    for q in QUANTITIES:
        if (doc.get("coverage") or {}).get(q) not in k["coverage"]["fields"][q]["values"]:
            out.append(f"{rid}: coverage {q} {(doc.get('coverage') or {}).get(q)!r} not allowed")
    for e in doc.get("defences") or []:
        for f in DEF_CATEGORICAL:
            if e.get(f) not in k["defences"]["fields"][f]["values"]:
                out.append(f"{rid}: {e.get('name')}: {f} {e.get(f)!r} not allowed")
        for f in DEF_MULTI:
            vals = e.get(f) or []
            if not vals or any(v not in k["defences"]["fields"][f]["values"] for v in vals):
                out.append(f"{rid}: {e.get('name')}: {f} {vals!r} not allowed")
        if not e.get("bases"):
            out.append(f"{rid}: {e.get('name')}: no base models listed")
    answered = [v["result"] for v in doc.get("values") or []]
    drawn = [c["result"] for c in cards]
    if sorted(answered) != sorted(drawn):
        out.append(f"{rid}: values answered {sorted(answered)} differ from the cards drawn {sorted(drawn)}")
    ids = {d["id"] for d in first.get("defences") or []}
    if len(match_line or []) != len(doc.get("defences") or []):
        out.append(f"{rid}: {len(match_line or [])} match lines for {len(doc.get('defences') or [])} defence entries")
    for ids_ in match_line or []:
        for i in ids_ or []:
            if i not in ids:
                out.append(f"{rid}: match {i!r} is not a first-pass defence")
    used = [i for ids_ in match_line or [] for i in ids_ or []]
    for i in {x for x in used if used.count(x) > 1}:
        out.append(f"{rid}: first-pass defence {i!r} matched twice")
    return out


# ---- agreement --------------------------------------------------------------------------------------------------

def cohen_kappa(pairs: list[tuple]) -> float | None:
    """Cohen's kappa over (first, second) pairs; missing values are a category. None when chance agreement is 1
    (every value in one category), where kappa is undefined and exact agreement is reported alone."""
    n = len(pairs)
    if not n:
        return None
    pairs = [(MISSING if a in (None, "") else a, MISSING if b in (None, "") else b) for a, b in pairs]
    po = sum(a == b for a, b in pairs) / n
    ca, cb = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    pe = sum(ca[c] * cb[c] for c in set(ca) | set(cb)) / (n * n)
    return None if pe >= 1 else (po - pe) / (1 - pe)


def jaccard(a, b) -> float:
    a, b = set(a or []), set(b or [])
    return 1.0 if not a and not b else len(a & b) / len(a | b)


def _number(s):
    t = str(s).strip().replace(",", "").replace("−", "-").replace("%", "").replace("+", "").strip()
    m = re.fullmatch(r"-?\d+(\.\d+)?", t)
    if not m:
        return None, None
    return float(t), len(m.group(1)) - 1 if m.group(1) else 0


def values_agree(first, second) -> bool:
    """Exact match within rounding: both printed numbers rounded to the coarser of their two precisions."""
    a, da = _number(first)
    b, db = _number(second)
    if a is None or b is None:
        return re.sub(r"\s+", "", str(first)) == re.sub(r"\s+", "", str(second))
    d = min(da, db)
    return round(a, d) == round(b, d)


def agreement(first: dict, second: dict, matches: dict) -> tuple[list[dict], list[dict], list[dict]]:
    """Agreement between the first coding (record files) and the second pass, before any disagreement is resolved.
    first: report id -> record; second: report id -> second-pass document; matches: report id -> list (one entry per
    second-pass defence, in order) of first-pass defence ids. Returns (field rows, per-report rows, disagreements)."""
    cat = defaultdict(list)          # field -> [(first, second)]
    multi = defaultdict(list)        # field -> [jaccard]
    multi_exact = defaultdict(list)
    vals, set_j, dis, per_report = [], [], [], []
    for rid in sorted(second):
        rec, sp, mt = first[rid], second[rid], matches[rid]
        cat["role"].append((rec["report"]["role"], sp["role"]))
        if rec["report"]["role"] != sp["role"]:
            dis.append({"report": rid, "unit": "report", "field": "role", "first": rec["report"]["role"], "second": sp["role"]})
        for q in QUANTITIES:
            a, b = rec["coverage"].get(q), sp["coverage"].get(q)
            cat[f"coverage.{q}"].append((a, b))
            if a != b:
                dis.append({"report": rid, "unit": "coverage", "field": q, "first": a, "second": b})
        dfs = {d["id"]: d for d in rec.get("defences") or []}
        if len(mt) != len(sp["defences"]):
            raise ValueError(f"{rid}: {len(mt)} match lines for {len(sp['defences'])} second-pass defences")
        matched = set()
        for entry, ids in zip(sp["defences"], mt):
            for did in ids or []:
                if did not in dfs:
                    raise ValueError(f"{rid}: match {did!r} is not a first-pass defence")
                matched.add(did)
                d = dfs[did]
                for f in DEF_CATEGORICAL:
                    cat[f"defence.{f}"].append((d.get(f), entry.get(f)))
                    if d.get(f) != entry.get(f):
                        dis.append({"report": rid, "unit": did, "field": f, "first": d.get(f), "second": entry.get(f)})
                for f in DEF_MULTI:
                    j = jaccard(d.get(f), entry.get(f))
                    multi[f"defence.{f}"].append(j)
                    multi_exact[f"defence.{f}"].append(set(d.get(f) or []) == set(entry.get(f) or []))
                    if j < 1:
                        dis.append({"report": rid, "unit": did, "field": f, "first": ";".join(d.get(f) or []),
                                    "second": ";".join(entry.get(f) or [])})
        # defence set at the method level: second-pass entries against first-pass groups; first-pass defences no
        # entry matched are grouped by name (with the base model removed) so one missed method counts once
        unmatched_first = [d for i, d in dfs.items() if i not in matched]
        groups = {re.sub(r"\s+on\s+.*$", "", d["name"]).strip().lower() for d in unmatched_first}
        n_entries = len(sp["defences"])
        n_matched_entries = sum(1 for ids in mt if ids)
        j = n_matched_entries / (n_entries + len(groups)) if (n_entries + len(groups)) else 1.0
        set_j.append(j)
        for d in unmatched_first:
            dis.append({"report": rid, "unit": d["id"], "field": "defence set", "first": d["name"], "second": "(not coded)"})
        for entry, ids in zip(sp["defences"], mt):
            if not ids:
                dis.append({"report": rid, "unit": "second pass", "field": "defence set", "first": "(not coded)", "second": entry["name"]})
        res = {r["id"]: r for r in rec.get("results") or []}
        n_ok = 0
        for v in sp.get("values") or []:
            r = res.get(v["result"])
            if r is None:
                raise ValueError(f"{rid}: value card {v['result']!r} is not a first-pass result")
            ok = values_agree(r["value"], v["value"])
            n_ok += ok
            vals.append(ok)
            if not ok:
                dis.append({"report": rid, "unit": v["result"], "field": "value", "first": r["value"], "second": v["value"]})
        per_report.append({"report": rid, "first_defences": len(dfs), "second_entries": n_entries,
                           "matched_entries": n_matched_entries, "unmatched_first_groups": len(groups),
                           "defence_set_jaccard": round(j, 3), "values": len(sp.get("values") or []), "values_agree": n_ok})
    rows = []

    def add(ftype, field, n, exact=None, kappa=None, jac=None):
        ok = True
        if exact is not None and exact < EXACT_MIN:
            ok = False
        if kappa is not None and kappa < KAPPA_MIN:
            ok = False
        if jac is not None and jac < JACCARD_MIN:
            ok = False
        rows.append({"field_type": ftype, "field": field, "pairs": n,
                     "exact_agreement": "" if exact is None else round(exact, 3),
                     "kappa": "" if kappa is None else round(kappa, 3),
                     "mean_jaccard": "" if jac is None else round(jac, 3),
                     "meets_threshold": "yes" if ok else "no", "action": "none" if ok else "re-code this field in full"})
    for f, pairs in cat.items():
        add("categorical", f, len(pairs), sum(a == b for a, b in pairs) / len(pairs), cohen_kappa(pairs))
    allc = [p for ps in cat.values() for p in ps]
    add("categorical (pooled)", "all categorical fields", len(allc), sum(a == b for a, b in allc) / len(allc), cohen_kappa(allc))
    rows[-1]["action"] = "summary row (the rule applies field by field)"
    for f, js in multi.items():
        add("multi-label", f, len(js), sum(multi_exact[f]) / len(js), None, sum(js) / len(js))
    add("multi-label", "defence set (method level, per report)", len(set_j), None, None, sum(set_j) / len(set_j))
    add("numeric", "printed values re-read from their addresses", len(vals), sum(vals) / len(vals) if vals else 0)
    return rows, per_report, dis


# ---------------------------------------------------------------- 04r: applying a listed re-code, and the amended rules as checks
CHANGE_FIELDS = ["report", "unit", "field", "old", "new", "reason", "page"]
LIST_FIELDS = {"axis1", "axis2", "also_quantity"}
PROMPT_OR_RULE = {"prompting", "fixed_inference_rule"}


def encode(v) -> str:
    """One cell for a field value: lists joined by ';', absent as ''."""
    if v is None:
        return ""
    return ";".join(str(x) for x in v) if isinstance(v, list) else str(v)


def _decode(field: str, s: str):
    if s == "":
        return None
    if field.split(".")[-1] in LIST_FIELDS:
        return s.split(";")
    if field.startswith("pages."):
        return int(s)
    return s


def _target(rec: dict, unit: str) -> dict:
    if unit in ("report", "coverage"):
        return rec[unit]
    kind, _, rid = unit.partition(":")
    for row in rec.get(kind) or []:
        if row.get("id") == rid:
            return row
    raise KeyError(unit)


def field_value(rec: dict, unit: str, field: str):
    obj = _target(rec, unit)
    if "." in field:
        a, b = field.split(".", 1)
        return (obj.get(a) or {}).get(b)
    return obj.get(field)


def apply_changes(records: dict, rows: list[dict]) -> list[str]:
    """Applies listed changes in place. A row whose old value is not the value on file is refused, so a change list
    built from other files than the committed ones cannot be applied by mistake."""
    problems = []
    for r in rows:
        rec = records.get(r["report"])
        if rec is None:
            problems.append(f"{r['report']}: no such record")
            continue
        try:
            obj = _target(rec, r["unit"])
        except KeyError:
            problems.append(f"{r['report']}:{r['unit']}: no such unit")
            continue
        cur = encode(field_value(rec, r["unit"], r["field"]))
        if cur != r["old"]:
            problems.append(f"{r['report']}:{r['unit']}:{r['field']} is {cur!r} on file, the change list expects {r['old']!r}")
            continue
        new = _decode(r["field"], r["new"])
        if "." in r["field"]:
            a, b = r["field"].split(".", 1)
            sub = obj.get(a) or {}
            if new is None:
                sub.pop(b, None)
            else:
                sub[b] = new
            if sub:
                obj[a] = sub
            else:
                obj.pop(a, None)
        elif new is None:
            obj.pop(r["field"], None)
        else:
            obj[r["field"]] = new
    return problems


def rule_problems(rec: dict) -> list[str]:
    """Amended rules 8, 9 and 32 (coding_rules.md, Amendment 2) checked on one record."""
    out = []
    has_none = any(str(e.get("defence_id", "")).startswith("none:") for e in rec.get("evaluations") or [])
    for d in rec.get("defences") or []:
        a1, sig = d["axis1"], d["signal_dependence"]
        trained = not set(d["axis2"]) <= PROMPT_OR_RULE
        tag = f"defences:{d['id']}"
        if "no_signal" in a1 and len(a1) > 1:
            out.append(f"{tag}: no_signal combined with another axis 1 value")
        if (a1 == ["no_signal"]) != (sig == "not_applicable"):
            out.append(f"{tag}: axis 1 {encode(a1)} with signal_dependence {sig} (not_applicable exactly when axis 1 is no_signal)")
        if trained and sig == "baseline_only":
            out.append(f"{tag}: baseline_only on a trained defence")
        if not trained and sig == "stated":
            out.append(f"{tag}: stated on a prompting defence or fixed rule (measured, baseline_only or not_reported)")
        if sig == "baseline_only" and not has_none:
            out.append(f"{tag}: baseline_only but the report has no undefended evaluation")
        if sig == "measured_other_checkpoint" and not d.get("signal_dependence_detail"):
            out.append(f"{tag}: measured_other_checkpoint without signal_dependence_detail naming the checkpoint")
        if d["origin"] == "other_report" and d["isolation"] == "no_ablation":
            out.append(f"{tag}: no_ablation on another report's defence (not_applicable unless this report ablates it)")
    cov = rec.get("coverage") or {}
    for q in ("R1", "R3ns", "R3cp", "R3safe"):
        rs = [x for x in rec.get("results") or [] if x.get("quantity") == q or q in (x.get("also_quantity") or [])]
        if cov.get(q) in ("direct", "proxy") and not rs:
            out.append(f"coverage {q} is {cov.get(q)} but no result carries {q}")
        if cov.get(q) in ("not_reported", "not_applicable") and rs:
            out.append(f"coverage {q} is {cov.get(q)} but {len(rs)} results carry {q}")
        main = [x["coverage"] for x in rs if x.get("quantity") == q]
        if main and len(main) == len(rs) and cov.get(q) in ("direct", "proxy"):
            best = "direct" if "direct" in main else "proxy"
            if best != cov.get(q):
                out.append(f"coverage {q} is {cov.get(q)} but the best result coded for it is {best}")
    return out

In [ ]:
%%writefile tests/test_extract.py
"""Tests for src/extract.py: the seeded sample, value cards, tables, cross-record checks and agreement statistics."""
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
extract = importlib.import_module("extract")
schema_mod = importlib.import_module("schema")


def _rec(rid="r_a", slug="s_a"):
    return {"report": {"id": rid, "slug": slug, "role": "proposes", "read": "F"},
            "coverage": {q: "not_reported" for q in extract.QUANTITIES} | {"R1": "direct"},
            "defences": [{"id": "d1", "name": "Def", "origin": "this_report", "base_checkpoint": "M", "axis1": ["chat_roles"],
                          "axis2": ["sft"], "isolation": "no_ablation", "signal_dependence": "not_reported",
                          "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2}],
            "evaluations": [{"id": "e1", "defence_id": "d1", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3},
                            {"id": "e0", "defence_id": "none:M", "benchmark": "B", "attacker_type": "static", "attacker_access": "black_box",
                             "judge": "J", "layout_same_as_defence": "yes", "measured_by": rid, "page": 3}],
            "results": [{"id": f"x{i}", "evaluation_id": "e1" if i % 2 else "e0", "quantity": "R1", "coverage": "direct",
                         "metric_as_printed": "ASR", "value": f"{i}.5", "unit": "percent", "direction": "lower_better",
                         "qualifier": "exact", "value_source": "table", "page": 3} for i in range(9)],
            "comparisons": [{"id": "c1", "left": "e1", "right": "e0", "purpose": "W", "varies": "defence",
                             **{c: "matched" for c in schema_mod.CONDITIONS}, "page": 3}],
            "claims": [{"id": "k1", "text": "t", "location": "abstract", "anchor": "a", "jobs": ["R1"], "supported_by": ["c1"], "page": 1}]}


def test_sample_is_stable_and_twenty_percent():
    slugs = [f"s{i:03d}" for i in range(124)]
    s = extract.second_pass_sample(slugs)
    assert len(s) == 25 and len(set(s)) == 25
    dropped = [x for x in slugs if x != slugs[-1] and x not in s[:1]]
    s2 = extract.second_pass_sample(dropped)
    assert set(s[1:]) <= set(s2)                       # removing one study does not reorder the others
    assert extract.second_pass_sample(slugs) == s      # deterministic


def test_cards_hide_values_and_are_capped():
    cards = extract.value_cards(_rec())
    assert len(cards) == extract.CARDS_PER_REPORT
    assert all("value" not in c for c in cards)
    assert {c["arm"].startswith(("Def on", "no defence")) for c in cards} == {True}


def test_tables_have_global_ids_and_grades():
    sch = schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml")
    t = extract.build_tables({"r_a": _rec()}, sch, schema_mod.grade)
    assert t["defences"][0][0]["defence_uid"] == "r_a:d1"
    ev = {r["evaluation_uid"]: r for r in t["evaluations"][0]}
    assert ev["r_a:e1"]["measurement"] == "original" and ev["r_a:e0"]["measurement"] == "undefended"
    assert t["comparisons"][0][0]["grade"] == "A"
    assert t["claims"][0][0]["supported_by"] == "r_a:c1"


def test_cross_record_checks():
    recs = {"r_a": _rec()}
    assert extract.cross_record_problems(recs, ["s_a"]) == []
    probs = extract.cross_record_problems(recs, ["s_a", "s_b"], {"r_a": 2})
    assert any("s_b has no record" in p for p in probs)
    assert any("beyond the PDF's 2 pages" in p for p in probs)


def test_kappa_jaccard_and_rounding():
    assert extract.cohen_kappa([("a", "a"), ("b", "b")]) == 1.0
    assert extract.cohen_kappa([("a", "a"), ("a", "a")]) is None
    assert abs(extract.cohen_kappa([("a", "a"), ("a", "b"), ("b", "b"), ("b", "a")])) < 1e-9
    assert extract.jaccard(["x", "y"], ["x"]) == 0.5
    assert extract.values_agree("45.20%", "45.2") and extract.values_agree("+0.028", "0.028")
    assert not extract.values_agree("45.2", "45.3") and extract.values_agree("12", "12.0")


def test_agreement_counts_disagreements():
    rec = _rec()
    sp = {"report": "r_a", "role": "proposes", "coverage": dict(rec["coverage"]),
          "defences": [{"name": "Def", "bases": ["M"], "origin": "this_report", "axis1": ["chat_roles", "position"], "axis2": ["sft"],
                        "isolation": "ablation_present", "signal_dependence": "not_reported", "trains_instruction_pair": "no",
                        "trains_carry_through": "no"}],
          "values": [{"result": "x1", "value": "1.5"}, {"result": "x2", "value": "9"}]}
    rows, per, dis = extract.agreement({"r_a": rec}, {"r_a": sp}, {"r_a": [["d1"]]})
    by = {r["field"]: r for r in rows}
    assert by["defence.isolation"]["exact_agreement"] == 0.0
    assert by["defence.axis1"]["mean_jaccard"] == 0.5
    assert by["printed values re-read from their addresses"]["exact_agreement"] == 0.5
    assert {d["field"] for d in dis} == {"isolation", "axis1", "value"}
    assert extract.validate_second_pass(sp, schema_mod.load_schema(Path(__file__).resolve().parent.parent / "schema" / "record_schema.yaml"),
                                        [{"result": "x1"}, {"result": "x2"}], rec, [["d1"]]) == []


def test_apply_changes_checks_the_old_value_and_edits_in_place():
    recs = {"r_a": _rec()}
    rows = [{"report": "r_a", "unit": "defences:d1", "field": "axis1", "old": "chat_roles", "new": "no_signal", "reason": "", "page": ""},
            {"report": "r_a", "unit": "defences:d1", "field": "pages.isolation", "old": "", "new": "5", "reason": "", "page": "5"},
            {"report": "r_a", "unit": "results:x1", "field": "also_quantity", "old": "", "new": "R3cp", "reason": "", "page": ""},
            {"report": "r_a", "unit": "coverage", "field": "R3cp", "old": "not_reported", "new": "proxy", "reason": "", "page": ""},
            {"report": "r_a", "unit": "coverage", "field": "R1", "old": "proxy", "new": "direct", "reason": "", "page": ""}]
    probs = extract.apply_changes(recs, rows)
    d = recs["r_a"]["defences"][0]
    assert d["axis1"] == ["no_signal"] and d["pages"] == {"isolation": 5}
    assert recs["r_a"]["results"][1]["also_quantity"] == ["R3cp"] and recs["r_a"]["coverage"]["R3cp"] == "proxy"
    assert len(probs) == 1 and "expects 'proxy'" in probs[0]          # R1 is direct on file: the row is refused
    back = [{"report": "r_a", "unit": "defences:d1", "field": "pages.isolation", "old": "5", "new": "", "reason": "", "page": ""}]
    assert extract.apply_changes(recs, back) == [] and "pages" not in d


def test_rule_problems_follow_amendment_2():
    rec = _rec()
    assert extract.rule_problems(rec) == []
    d = rec["defences"][0]
    d["axis1"], d["signal_dependence"] = ["no_signal"], "baseline_only"
    probs = extract.rule_problems(rec)
    assert any("not_applicable exactly when" in p for p in probs) and any("baseline_only on a trained" in p for p in probs)
    d["axis1"], d["axis2"], d["signal_dependence"] = ["delimiters_markers"], ["prompting"], "stated"
    assert any("stated on a prompting" in p for p in extract.rule_problems(rec))
    d["signal_dependence"], d["origin"], d["isolation"] = "baseline_only", "other_report", "no_ablation"
    assert any("another report's defence" in p for p in extract.rule_problems(rec))
    d["origin"] = "this_report"
    rec["coverage"]["R3cp"] = "proxy"
    assert any("coverage R3cp is proxy but no result" in p for p in extract.rule_problems(rec))

In [ ]:
import schema, search, paths, extract
for m in (schema, search, paths, extract):
    importlib.reload(m)
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_extract.py"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout.strip() else t.stderr.strip()[-400:])

## Stage 3. The re-code: 525 changed fields, each with its old value, new value, reason and page

The file lists only fields whose value changes. A row is applied only if its old value is the value on file. Before and after, every record file is checked against a fixed SHA-256 digest, so the result is the same file set on any machine.

In [ ]:
%%writefile reports/recode_04r_changes.csv
report,unit,field,old,new,reason,page
actconsistency_arxiv2026,defences:act_q17,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:bct_q17,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:act_q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:bct_q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:act_phi,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:bct_phi,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:act_oss,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:bct_oss,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:act_gem,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:bct_gem,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
actconsistency_arxiv2026,defences:secalign_q8,axis1,no_signal,delimiters_markers,the report describes SecAlign as extending StruQ's structured queries that delimit instructions from data; no_signal would say SecAlign shows no source,3
adaptiveipi_arxiv2025,defences:ip_v,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
adaptiveipi_arxiv2025,defences:ip_l,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
advershield_jtsi2026,defences:tagd,axis1,internal_intervention,no_signal,"TAGD scales its noise by attention salience, not by the source of the token; no source is shown at inference",3
agentsafeevo_arxiv2026,defences:spot,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",8
asb_iclr2025,defences:instr_m00,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m01,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m02,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m03,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m04,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m05,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m06,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m07,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m08,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m09,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m10,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m11,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
asb_iclr2025,defences:instr_m12,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
attackdefence_arxiv2024,defences:ign_l3,isolation,not_applicable,no_ablation,"a defence this report runs as a method; no run with a component removed or changed is reported (Section 5.4 tests closed models, gradient attacks and StruQ, not components)",
attackdefence_arxiv2024,defences:fct_l3,isolation,not_applicable,ablation_present,"Ours-Fakecom, the same defence without the chat template, is reported beside Fakecom-t (Table 3)",7
attackdefence_arxiv2024,defences:fct_l3,pages.isolation,,7,page of the fact behind the new isolation,7
attackdefence_arxiv2024,defences:ign_q2,isolation,not_applicable,no_ablation,"a defence this report runs as a method; no run with a component removed or changed is reported (Section 5.4 tests closed models, gradient attacks and StruQ, not components)",
attackdefence_arxiv2024,defences:fct_q2,isolation,not_applicable,ablation_present,"Ours-Fakecom, the same defence without the chat template, is reported beside Fakecom-t (Table 3)",7
attackdefence_arxiv2024,defences:fct_q2,pages.isolation,,7,page of the fact behind the new isolation,7
attackdefence_arxiv2024,defences:ign_l31,isolation,not_applicable,no_ablation,"a defence this report runs as a method; no run with a component removed or changed is reported (Section 5.4 tests closed models, gradient attacks and StruQ, not components)",
attackdefence_arxiv2024,defences:fct_l31,isolation,not_applicable,ablation_present,"Ours-Fakecom, the same defence without the chat template, is reported beside Fakecom-t (Table 3)",7
attackdefence_arxiv2024,defences:fct_l31,pages.isolation,,7,page of the fact behind the new isolation,7
autodojo_arxiv2026,defences:rem_g4m,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
autodojo_arxiv2026,defences:rem_g54m,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
autodojo_arxiv2026,defences:rem_gem,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
autodojo_arxiv2026,defences:rem_ds,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
autodojo_arxiv2026,defences:rem_hk,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
autonomytax_arxiv2026,defences:secalign_l3,axis1,no_signal,delimiters_markers,"the report gives SecAlign's training but not its layout (p.12); SecAlign builds on StruQ's delimited format, which this report codes delimiters_markers; no_signal would say SecAlign shows no source",12
autonomytax_arxiv2026,defences:secalign_mi,axis1,no_signal,delimiters_markers,"the report gives SecAlign's training but not its layout (p.12); SecAlign builds on StruQ's delimited format, which this report codes delimiters_markers; no_signal would say SecAlign shows no source",12
backfire_zenodo2026,defences:srp_llama,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
backfire_zenodo2026,defences:spot_llama,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
backfire_zenodo2026,defences:srp_ds,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
backfire_zenodo2026,defences:spot_ds,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
backfire_zenodo2026,defences:srp_qwen,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
backfire_zenodo2026,defences:spot_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
backfire_zenodo2026,defences:srp_haiku,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
backfire_zenodo2026,defences:spot_haiku,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
backfire_zenodo2026,defences:srp_sonnet,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
backfire_zenodo2026,defences:spot_sonnet,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
benchvalid_rs2026,defences:instr_gemma2,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_gemma2,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_llama33,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_llama33,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_dsr1,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_dsr1,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_phi4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_phi4,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_novapro,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_novapro,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_pixtral,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_pixtral,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_qwen14,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_qwen14,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_mistral7,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_mistral7,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_llama31,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_llama31,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
benchvalid_rs2026,defences:instr_qwen7,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
benchvalid_rs2026,defences:spot_qwen7,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
cacheprune_arxiv2025,defences:cp_l3,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
cacheprune_arxiv2025,defences:cp_m7,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
cacheprune_arxiv2025,defences:cp_phi,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
civ_arxiv2025,defences:civ_l3,signal_dependence,measured,baseline_only,"the Table 3 variants change how the trust labels are signed, propagated or cached; none runs CIV without the labels; the signal-removed condition is the frozen undefended model, which is reported",9
civ_arxiv2025,defences:civ_l3,signal_dependence_detail,"Table 3: unsigned labels give 7.1% ASR and no trust propagation 3.2%, against 0% for the full system","Table 3 variants change how the labels are signed, propagated or cached and keep them; the signal-removed condition is the frozen undefended model",detail follows the new signal_dependence,
civ_arxiv2025,defences:civ_unsigned_l3,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",9
codel_arxiv2026,defences:agentalign_q,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
codel_arxiv2026,defences:reta_q,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
codel_arxiv2026,defences:agentalign_l,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
codel_arxiv2026,defences:reta_l,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_0,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_0,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_1,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_1,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_2,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_2,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_3,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_5,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_5,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_6,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_6,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_7,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_7,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_8,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_9,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_9,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_10,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_10,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_11,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_11,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_12,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_12,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_13,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_13,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_14,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_14,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_15,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_15,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_16,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_16,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
codt_arxiv2025,defences:codt_17,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
codt_arxiv2025,defences:codt_17,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
constsanitlog_icssa2026,defences:dm_q,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",5
copa_arxiv2026,defences:secalign_l31,axis1,no_signal,delimiters_markers,the report does not describe SecAlign's layout; coded as the other unspecified re-runs of SecAlign in the corpus are coded; no_signal would say SecAlign shows no source,
countersteer_arxiv2026,defences:cs_gptoss20b,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
countersteer_arxiv2026,defences:cs_qwen3_30b,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
countersteer_arxiv2026,defences:cs_llama31_8b,signal_dependence,stated,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
countersteer_arxiv2026,defences:spot_gptoss20b,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
crossdt_sbseg2026,defences:dt_llama,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
crossdt_sbseg2026,defences:dt_qwen,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
crossdt_sbseg2026,defences:dt_phi,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
dynppa_arxiv2026,defences:ppa_l33,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
dynppa_arxiv2026,defences:dyn_l33,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
eroded_zenodo2026,defences:msaft2026,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (the released Meta-SecAlign after the report's own benign fine-tuning; the defence's components are not removed),
eroded_zenodo2026,defences:msaft2027,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (the released Meta-SecAlign after the report's own benign fine-tuning; the defence's components are not removed),
eroded_zenodo2026,defences:msaft2028,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (the released Meta-SecAlign after the report's own benign fine-tuning; the defence's components are not removed),
eroded_zenodo2026,defences:reponly,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (the repair applied to the undefended model, a variant of ftrep)",
explosive_arxiv2026,defences:epdpo,isolation,not_applicable,no_ablation,"a defence this report runs as a method; no run with a component removed or changed is reported (the reported steps are checkpoints, not components)",
gemini_arxiv2025,defences:icl_g20,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
glasswall_techrxiv2026,defences:mw,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",5
gptred_arxiv2026,defences:gpt55,isolation,no_ablation,not_applicable,"the record is itself a variant or control of another record in the same report (the previous production model, the comparison arm for GPT-Red training)",
haaulc_rs2026,defences:ulc,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
haaulc_rs2026,defences:ulc,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
hybridrag_ijaiml2026,defences:san_qw,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
hybridrag_ijaiml2026,defences:ih_qw,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
hybridrag_ijaiml2026,defences:hyb_qw,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
hybridrag_ijaiml2026,defences:san_phi,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
hybridrag_ijaiml2026,defences:ih_phi,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
hybridrag_ijaiml2026,defences:hyb_phi,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
hybridrag_ijaiml2026,defences:san_mis,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
hybridrag_ijaiml2026,defences:ih_mis,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
hybridrag_ijaiml2026,defences:hyb_mis,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
icon_arxiv2026,defences:icon_q3,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
icon_arxiv2026,defences:icon_q3,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
icon_arxiv2026,defences:icon_l31,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
icon_arxiv2026,defences:icon_l31,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
icon_arxiv2026,defences:icon_m8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
icon_arxiv2026,defences:icon_m8,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
ignorednotrefused_arxiv2026,defences:inter_qwen9,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_qwen9,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_qwen9,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_qwen9,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_gemma4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_gemma4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_gemma4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_gemma4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_spark4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_spark4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_ornith9,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_ornith9,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_ornith9,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_ornith9,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_qwen4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_qwen4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_qwen4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_qwen4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_gemma12,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_gemma12,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_gemma12,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_gemma12,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_nemo4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_nemo4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_nemo4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_nemo4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_nanb3,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_nanb3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_nanb3,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_nanb3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_minicpm2,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_minicpm2,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_minis14,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_minis14,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_minis14,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_minis14,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:inter_lfm26,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:inter_lfm26,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ignorednotrefused_arxiv2026,defences:guard_lfm26,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ignorednotrefused_arxiv2026,defences:guard_lfm26,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
ihdec_arxiv2026,defences:ihdec_l31,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
instrucot_arxiv2026,defences:ip_l3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
instrucot_arxiv2026,defences:ip_l31,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
instrucot_arxiv2026,defences:ip_q25,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
instrucot_arxiv2026,defences:ip_q3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
intentguard_arxiv2025,defences:ig_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
intentguard_arxiv2025,defences:sep_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
intentguard_arxiv2025,defences:ig_oss,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
intentguard_arxiv2025,defences:sep_oss,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
ipidefrepl_computers2026,defences:spot_g54,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",19
ipidefrepl_computers2026,defences:spot_g54m,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",19
ipidefrepl_computers2026,defences:spot_cs46,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",19
ise_iclr2025,defences:advtext_l2_13b,signal_dependence,not_applicable,not_reported,a trained defence with a source signal; no run with the signal removed or moved,
jatmohouyi_arxiv2025,defences:jatmo_q05,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
jatmohouyi_arxiv2025,defences:jatmo_q4,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
jatmohouyi_arxiv2025,defences:jatmo_l7,isolation,no_ablation,not_applicable,"a defence from another report, run without removing or changing its components",
ldd_arxiv2025,defences:ldd_m0,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m1,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m2,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m3,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m5,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m6,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m7,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
ldd_arxiv2025,defences:ldd_m8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
liu_benchmark_arxiv2023,defences:instr_gpt4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
localrag_applsci2026,defences:c3,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
localrag_applsci2026,defences:c4,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
loginject_arxiv2026,defences:spot,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
loginject_arxiv2026,report,role,mixed,benchmark,the report's contribution is measurement on its log-injection benchmark and an evaluation of existing defence layers (p.19); it proposes no defence of its own,19
longpibench_arxiv2026,defences:instr,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
longpibench_arxiv2026,defences:delim,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",8
medmas_arxiv2026,defences:instr_sq,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
medmas_arxiv2026,defences:delim_sq,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
medmas_arxiv2026,defences:instr_mq,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
medmas_arxiv2026,defences:delim_mq,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
medmas_arxiv2026,defences:instr_aq,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
medmas_arxiv2026,defences:delim_aq,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
medmas_arxiv2026,defences:instr_sl,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
medmas_arxiv2026,defences:delim_sl,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
medmas_arxiv2026,defences:instr_ml,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
medmas_arxiv2026,defences:delim_ml,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
municipal_sauc2026,defences:dm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",9
municipal_sauc2026,defences:sq,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",9
municipal_sauc2026,coverage,R1,direct,aggregate_only,every result is pooled over the models (the defences' base is a pooled set); aggregate_only applies whatever the instrument,7
municipal_sauc2026,coverage,R2clean,direct,aggregate_only,follows from the same reading as R1: pooled over the models,7
peerrevcheck_osf2025,defences:check,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
peerrevcheck_osf2025,defences:check,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
peerrevcheck_osf2025,coverage,R1,proxy,aggregate_only,every result is pooled over the models (the defences' base is a pooled set); aggregate_only applies whatever the instrument,4
peerrevcheck_osf2025,coverage,R2clean,proxy,aggregate_only,follows from the same reading as R1: pooled over the models,4
pictionary_arxiv2026,defences:pic_opus,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_opus,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_opus,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_haiku,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_haiku,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_haiku,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_g55,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_g55,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_g55,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_nano,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_nano,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_nano,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_mini,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_mini,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_mini,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_qwen,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_qwen,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_qwen,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_kimi,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_kimi,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_kimi,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_grok,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_grok,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_grok,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_gpro,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_gpro,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_gpro,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_glite,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_glite,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_glite,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
pictionary_arxiv2026,defences:pic_g54,isolation,not_applicable,ablation_present,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles)",19
pictionary_arxiv2026,defences:pic_g54,pages.isolation,,19,page of the fact behind the new isolation,19
pictionary_arxiv2026,defences:pic_g54,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
pictionary_arxiv2026,defences:aud_gaud,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",8
pictionary_arxiv2026,defences:aud_qomni,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",8
pidefthesis_upv2026,defences:spot_nano,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_nano,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_gem,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_gem,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_q7,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_q7,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_q32,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_q32,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_l3,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_l3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_l8,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_l8,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pidefthesis_upv2026,defences:spot_l70,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",34
pidefthesis_upv2026,defences:rem_l70,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pipette_arxiv2026,defences:pip_q4,axis1,chat_roles,no_signal,"adversarial training inside the AgentDojo and ASB tool-calling harness; the tool role is the harness's own; the method adds no role or turn, trains no order over roles and reads no role at inference",1
pipette_arxiv2026,defences:pip_q4,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
pipette_arxiv2026,defences:pip_l8,axis1,chat_roles,no_signal,"adversarial training inside the AgentDojo and ASB tool-calling harness; the tool role is the harness's own; the method adds no role or turn, trains no order over roles and reads no role at inference",1
pipette_arxiv2026,defences:pip_l8,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
pipette_arxiv2026,defences:pip_q27,axis1,chat_roles,no_signal,"adversarial training inside the AgentDojo and ASB tool-calling harness; the tool role is the harness's own; the method adds no role or turn, trains no order over roles and reads no role at inference",1
pipette_arxiv2026,defences:pip_q27,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
pipette_arxiv2026,defences:pip_oss,axis1,chat_roles,no_signal,"adversarial training inside the AgentDojo and ASB tool-calling harness; the tool role is the harness's own; the method adds no role or turn, trains no order over roles and reads no role at inference",1
pipette_arxiv2026,defences:pip_oss,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
pipette_arxiv2026,defences:spot_q4,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
pipette_arxiv2026,defences:spot_l8,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
pismith_colm2026,defences:instr_q,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
poisonedtools_zenodo2026,defences:d1,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
poisonedtools_zenodo2026,defences:d4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
poisonedtools_zenodo2026,coverage,R1,direct,aggregate_only,every result is pooled over the models (the defences' base is a pooled set); aggregate_only applies whatever the instrument,9
poisonedtools_zenodo2026,coverage,R2clean,direct,aggregate_only,follows from the same reading as R1: pooled over the models,9
ppa_arxiv2025,defences:ppa_g35,isolation,not_applicable,ablation_present,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported",5
ppa_arxiv2025,defences:ppa_g35,pages.isolation,,5,page of the fact behind the new isolation,5
ppa_arxiv2025,defences:ppa_g4,isolation,not_applicable,ablation_present,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported",5
ppa_arxiv2025,defences:ppa_g4,pages.isolation,,5,page of the fact behind the new isolation,5
ppa_arxiv2025,defences:ppa_l3,isolation,not_applicable,ablation_present,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported",5
ppa_arxiv2025,defences:ppa_l3,pages.isolation,,5,page of the fact behind the new isolation,5
ppa_arxiv2025,defences:ppa_ds,isolation,not_applicable,ablation_present,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported",5
ppa_arxiv2025,defences:ppa_ds,pages.isolation,,5,page of the fact behind the new isolation,5
provtransformer_arxiv2026,defences:nob_smol,signal_dependence,not_applicable,not_reported,a trained defence with a source signal; no run with the signal removed or moved,
provtransformer_arxiv2026,defences:nob_l3,signal_dependence,not_applicable,not_reported,a trained defence with a source signal; no run with the signal removed or moved,
provtransformer_arxiv2026,defences:nob_q25,signal_dependence,not_applicable,not_reported,a trained defence with a source signal; no run with the signal removed or moved,
provtransformer_arxiv2026,defences:nob_mi,signal_dependence,not_applicable,not_reported,a trained defence with a source signal; no run with the signal removed or moved,
provtransformer_arxiv2026,defences:instr_l3,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
pvst_arxiv2026,defences:rftsucc,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:rftgate,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:dpoany,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:dpoclean,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:dpooracle,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:rftgate_lo,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
pvst_arxiv2026,defences:dpogate_lo,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (stages, controls and training-subset variants of DPO-gate, the PVST method)",
ragmitig_ccs2026,defences:prompt,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",3
ragmitig_ccs2026,defences:iso,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",3
ragmitig_ccs2026,defences:ih,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
ragmitig_ccs2026,defences:ground,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
ragmitig_ccs2026,defences:flags,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",3
readable_arxiv2026,defences:match_slack,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:laf_slack,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:laf_slack,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:match_ws,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:laf_ws,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:laf_ws,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:ctrl_slack,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:ctrl_slack,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:rand_slack,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:rand_slack,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:l8_slack,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:l8_slack,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:match_llama,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:ctrl_llama,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:ctrl_llama,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:rand_llama,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:rand_llama,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:role_q7,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:role_q7,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
readable_arxiv2026,defences:role_q15,axis1,internal_intervention,no_signal,"the direction edit is applied to the residual stream at the final input token, wherever the untrusted text sits; it is not keyed by source",16
readable_arxiv2026,defences:role_q15,isolation,ablation_present,not_applicable,"the record is itself a variant or control of another record in the same report (leave-family-out, controlled-role, random-direction and off-band-layer variants of the suite-matched edit)",
reta_arxiv2026,defences:reta_q,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
reta_arxiv2026,defences:reta_l,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
rui_preprints2026,defences:stat_gfm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
rui_preprints2026,defences:rand_gfm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
rui_preprints2026,defences:stat_gfl,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
rui_preprints2026,defences:rand_gfl,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
rui_preprints2026,defences:rui_gfl,signal_dependence,measured,measured_other_checkpoint,the run with the privileged channel moved to the data (nopc_gfm) is on Gemini 3 Flash (medium thinking) only,
rui_preprints2026,defences:rui_gfl,signal_dependence_detail,,"nopc_gfm: privileged channel moved to the data, on Gemini 3 Flash (medium thinking) only",detail follows the new signal_dependence,
rui_preprints2026,defences:stat_gp,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rand_gp,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rui_gp,signal_dependence,measured,measured_other_checkpoint,the run with the privileged channel moved to the data (nopc_gfm) is on Gemini 3 Flash (medium thinking) only,
rui_preprints2026,defences:rui_gp,signal_dependence_detail,,"nopc_gfm: privileged channel moved to the data, on Gemini 3 Flash (medium thinking) only",detail follows the new signal_dependence,
rui_preprints2026,defences:stat_glm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rand_glm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rui_glm,signal_dependence,measured,measured_other_checkpoint,the run with the privileged channel moved to the data (nopc_gfm) is on Gemini 3 Flash (medium thinking) only,
rui_preprints2026,defences:rui_glm,signal_dependence_detail,,"nopc_gfm: privileged channel moved to the data, on Gemini 3 Flash (medium thinking) only",detail follows the new signal_dependence,
rui_preprints2026,defences:stat_g54,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rand_g54,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",21
rui_preprints2026,defences:rui_g54,signal_dependence,measured,measured_other_checkpoint,the run with the privileged channel moved to the data (nopc_gfm) is on Gemini 3 Flash (medium thinking) only,
rui_preprints2026,defences:rui_g54,signal_dependence_detail,,"nopc_gfm: privileged channel moved to the data, on Gemini 3 Flash (medium thinking) only",detail follows the new signal_dependence,
rui_preprints2026,defences:nopc_gfm,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (ablations of RUI),
rui_preprints2026,defences:noeai_gfm,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (ablations of RUI),
rvr_arxiv2025,defences:rvr_l3,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
rvr_arxiv2025,defences:rvr_q2,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
rvr_arxiv2025,defences:rvr_l31,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
safereview_arxiv2026,defences:sysdef,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
secfid_arxiv2026,defences:deftokens_llama31_8b_sf,axis1,learned_token_features,chat_roles,"DefensiveTokens are placed before the whole input and mark no source; the method's source signal is its instruction-in-system, data-in-user layout (defensivetokens_aisec2025, p.7), as coded for the other DefensiveTokens records",defensivetokens_aisec2025 p.7
secinfer_arxiv2025,defences:instr_l31,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
secopd_emnlp2026,defences:secopd,isolation,not_applicable,no_ablation,"a defence this report runs as a method; no run with a component removed or changed is reported (the GRPO baseline is another training method, not SecOPD with a part removed)",
securecai_arxiv2026,defences:secai_nosan,signal_dependence,not_reported,not_applicable,axis 1 is no_signal: no inference-time source signal,
securecai_arxiv2026,defences:cai,axis2,sft,sft;preference_optimisation,the report describes Constitutional AI as supervised critique-revision followed by RL from AI feedback (p.1); the RL stage is preference optimisation,1
securecai_arxiv2026,defences:ih,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
smallagentsq_zenodo2026,defences:inline_l1q4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_l1q4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
smallagentsq_zenodo2026,defences:spot_l1q4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_l1q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_l1q8,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
smallagentsq_zenodo2026,defences:spot_l1q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_q15q4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_q15q4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
smallagentsq_zenodo2026,defences:spot_q15q4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_q15q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smallagentsq_zenodo2026,defences:inline_q15q8,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
smallagentsq_zenodo2026,defences:spot_q15q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smalllocal_zenodo2026,defences:spot,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",15
smalllocal_zenodo2026,defences:rule,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
smalllocal_zenodo2026,defences:rule,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:warn_gemma,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:xml_gemma,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:spot_gemma,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:warn_llama,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:xml_llama,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:spot_llama,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:warn_phi,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:xml_phi,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:spot_phi,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:warn_mistral,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:xml_mistral,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:spot_mistral,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:warn_qwen,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
socportab_zenodo2026,defences:xml_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
socportab_zenodo2026,defences:spot_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",6
spotguard_applsci2026,defences:spot,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",19
spotlighting_arxiv2024,defences:spot_datamark_gpt4,signal_dependence,baseline_only,not_reported,prompting or fixed inference rule with a source signal; no undefended run of this base is reported,
tab_arxiv2026,defences:spot_mm,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",24
tab_arxiv2026,defences:spot_qw,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",24
tab_arxiv2026,defences:spot_g54,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",24
tab_arxiv2026,coverage,R1,proxy,direct,distractor execution is the rate at which the injected instruction is executed; R1 does not require the instruction to be malicious,24
tab_arxiv2026,coverage,R3cp,not_reported,proxy,"task resolution needs the agent to act on the helpful cue placed in the file it reads, instruction-bearing data the task asks for",24
tab_arxiv2026,coverage,where.R3cp,,Table 10 task resolution (the helpful cue in the file must be used),names the instrument for the new code,24
tab_arxiv2026,results:r_mm_base_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_mm_base_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_mm_spot_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_mm_spot_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_mm_rup_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_mm_rup_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_qw_base_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_qw_base_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_qw_spot_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_qw_spot_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_qw_rup_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_qw_rup_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_g54_base_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_g54_base_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_g54_spot_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_g54_spot_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
tab_arxiv2026,results:r_g54_rup_dexec,coverage,proxy,direct,R1 re-checked: execution rate of the injected instruction,24
tab_arxiv2026,results:r_g54_rup_res,also_quantity,,R3cp,R3cp re-checked: task resolution needs the helpful cue in the file,24
textpixels_arxiv2026,defences:render_qwen,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
textpixels_arxiv2026,defences:render_gpt,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
textpixels_arxiv2026,defences:render_haiku,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
textpixels_arxiv2026,defences:render_gem,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
textpixels_arxiv2026,defences:render_glm,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
textpixels_arxiv2026,defences:render_ds,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",4
tracer_emnlp2026,defences:rag7e1,isolation,ablation_present,not_applicable,the record is itself a variant or control of another record in the same report (the epoch-1 checkpoint of the 7B RAG specialist),
tracer_emnlp2026,defences:rag7e1,pages.isolation,8,,the old page supported the old isolation,
tracer_emnlp2026,defences:rag7e1,signal_dependence,not_reported,measured_other_checkpoint,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only,18
tracer_emnlp2026,defences:rag7e1,pages.signal_dependence,,18,page of the fact behind the new signal_dependence,18
tracer_emnlp2026,defences:rag7e1,signal_dependence_detail,,"Table 11: RAG wrapper and RAG system prompt removed on the 3B RAG specialist (rag3), not on this checkpoint",detail follows the new signal_dependence,
tracer_emnlp2026,defences:rag7,signal_dependence,not_reported,measured_other_checkpoint,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only,18
tracer_emnlp2026,defences:rag7,pages.signal_dependence,,18,page of the fact behind the new signal_dependence,18
tracer_emnlp2026,defences:rag7,signal_dependence_detail,,"Table 11: RAG wrapper and RAG system prompt removed on the 3B RAG specialist (rag3), not on this checkpoint",detail follows the new signal_dependence,
tracer_emnlp2026,defences:t6,isolation,not_applicable,ablation_present,the TRACER method itself on an unnamed checkpoint; its components are ablated (Tables 8 and 11),8
tracer_emnlp2026,defences:t6,pages.isolation,,8,page of the fact behind the new isolation,8
tracer_emnlp2026,defences:t6,signal_dependence,not_reported,measured_other_checkpoint,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only,18
tracer_emnlp2026,defences:t6,pages.signal_dependence,,18,page of the fact behind the new signal_dependence,18
tracer_emnlp2026,defences:t6,signal_dependence_detail,,"Table 11: RAG wrapper and RAG system prompt removed on the 3B RAG specialist (rag3), not on this checkpoint",detail follows the new signal_dependence,
tracer_emnlp2026,coverage,R2attack,proxy,direct,evidence accuracy scores the user's question under an injection; the report-level code disagreed with the coding of its own results (direct),16
tracer_emnlp2026,coverage,R3cp,not_reported,proxy,Table 6 scores answers that must use valid evidence carrying an embedded injection,16
tracer_emnlp2026,coverage,where.R3cp,,Table 6 evidence accuracy (valid evidence carrying an injection),names the instrument for the new code,16
tracer_emnlp2026,results:r_t6_rag_acc,also_quantity,,R3cp,R3cp re-checked: valid evidence carrying an injection must be used,16
tracer_emnlp2026,results:r_t6_tool_acc,also_quantity,,R3cp,R3cp re-checked: valid evidence carrying an injection must be used,16
tracer_emnlp2026,results:r_t6_agent_acc,also_quantity,,R3cp,R3cp re-checked: valid evidence carrying an injection must be used,16
transfact_zenodo2026,defences:srp_haiku,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
transfact_zenodo2026,defences:spot_haiku,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
transfact_zenodo2026,defences:srp_sonnet,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
transfact_zenodo2026,defences:spot_sonnet,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
transfact_zenodo2026,defences:srp_llama,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
transfact_zenodo2026,defences:spot_llama,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
transfact_zenodo2026,defences:srp_ds,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
transfact_zenodo2026,defences:spot_ds,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
transfact_zenodo2026,defences:srp_qwen,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
transfact_zenodo2026,defences:spot_qwen,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",12
univdef_arxiv2026,defences:delim_gem,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
univdef_arxiv2026,defences:delim_qw,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
univdef_arxiv2026,defences:delim_l3,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
univdef_arxiv2026,defences:delim_l31,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
univdef_arxiv2026,defences:delim_g35,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
univdef_arxiv2026,defences:delim_g4,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",7
untrustedcomponent_arxiv2026,defences:d1_15,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
untrustedcomponent_arxiv2026,defences:d1_7,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
untrustedcomponent_arxiv2026,defences:g1_15,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
verih_arxiv2025,defences:guard_q4,axis1,chat_roles,no_signal,SysHint and GuardRules are wording in the system prompt; the system and user roles are the template's own; prompt wording alone is no_signal,4
verih_arxiv2025,defences:guard_q4,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
verih_arxiv2025,defences:guard_q8,axis1,chat_roles,no_signal,SysHint and GuardRules are wording in the system prompt; the system and user roles are the template's own; prompt wording alone is no_signal,4
verih_arxiv2025,defences:guard_q8,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
verih_arxiv2025,defences:guard_q14,axis1,chat_roles,no_signal,SysHint and GuardRules are wording in the system prompt; the system and user roles are the template's own; prompt wording alone is no_signal,4
verih_arxiv2025,defences:guard_q14,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
verih_arxiv2025,defences:guard_q36,axis1,chat_roles,no_signal,SysHint and GuardRules are wording in the system prompt; the system and user roles are the template's own; prompt wording alone is no_signal,4
verih_arxiv2025,defences:guard_q36,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
verih_arxiv2025,defences:guard_phi,axis1,chat_roles,no_signal,SysHint and GuardRules are wording in the system prompt; the system and user roles are the template's own; prompt wording alone is no_signal,4
verih_arxiv2025,defences:guard_phi,signal_dependence,baseline_only,not_applicable,axis 1 is no_signal: no inference-time source signal,
whosaidthis_zenodo2026,defences:lever_q8,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_q8,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_q17,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_q17,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_smol,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_smol,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_gran,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_gran,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_olmo7,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_olmo7,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_phi4,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_phi4,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",14
whosaidthis_zenodo2026,defences:lever_q14,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_q14,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosaidthis_zenodo2026,defences:lever_q32,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_q32,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosaidthis_zenodo2026,defences:lever_fal,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_fal,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosaidthis_zenodo2026,defences:lever_yi,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_yi,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosaidthis_zenodo2026,defences:lever_olmo13,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_olmo13,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosaidthis_zenodo2026,defences:lever_olmo32,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosaidthis_zenodo2026,defences:lever_olmo32,signal_dependence,not_reported,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosewords_zenodo2026,defences:rolesteer,isolation,not_applicable,no_ablation,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched),
whosewords_zenodo2026,defences:rolesteer,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16
whosewords_zenodo2026,defences:randdir,signal_dependence,not_applicable,baseline_only,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported",16

In [ ]:
import copy, csv, yaml, datetime as dt
from collections import Counter
EXPECT_BEFORE = {'actconsistency_arxiv2026': '0e7e6ab4a33bdf643c7b3fdcf258198d8a4498696a780eb866886b48bf9e424e', 'adaptiveipi_arxiv2025': 'c575a093765f907eb47d8c6db559b3c400e5cc36769e237b3e5a73ba1d4332ec', 'advershield_jtsi2026': 'df74dd1150cb906f370a14927684220fdbe523f6425dbcad0f0fb1807b907688', 'advft2022_arxiv2025': '61752fde24964d3a06ad5e497fe69d4778bbf5ee04267a4b9c3f90925a620e59', 'advsim2real_arxiv2026': 'e1d9e6828069c1d2b6ee2d253c967ab1926d53f4e0ca6e82b4ce44fd76353539', 'agentdojo_neurips2024': '579ba1ec91ed348d2fddc0dac159dbaade07d5aba802f7cec740f36e5c4c02e5', 'agentdyn_arxiv2026': 'd0d32b665213b24b8438fe6b7febd0d203c3bf653a6cb04c1747062d8940d0bd', 'agentsafeevo_arxiv2026': '67765937b1c95d4646e6909a0d384254f710d0cee83f0c99675be418d5f1adb2', 'air_arxiv2025': 'e45f5b78f65f13888b7fc3a1a240f0472cbbfc444467b23bb8d100dcf8ecdf29', 'alwaysfall_arxiv2026': 'aa78b1618af7c62dd39a6063e89237c6aa275b7139738d224b5d413731a97771', 'aort_etrij2026': '1da167f5de0dcdfa3af3e1ad085c3e58c757d2729ae4431c3ca2128db44791b2', 'arlas_arxiv2025': '65f0443c9a7e26f2b945ea3657d4717e0b9b29c6c3f28fb59c4f9e182e3d086a', 'asb_iclr2025': '2adb9b2720d33b7c200f2cc3f31bab6437b12bd4f05a0cfaa420417ee96d97fa', 'aside_iclr2026': '1e8b5e8480a51239e79e03e7b39400d7e660897795cb0669be8a39b8ead6cad4', 'astra_arxiv2025': 'a21a4a3cf811be3cf5fa7f8b46992cabb1ac30c8f160bf4a854a75c584e4a8e3', 'attackdefence_arxiv2024': '66fd14893f1d7f25d229c4787e6c2919b74ac3dacc67d028b72af930dc662523', 'attackermovessecond_arxiv2025': 'c315d262660a2caa982fc9af3d38270a9d689b6363e4db02a8e61855861b6c20', 'autodojo_arxiv2026': '8445a5e9b33107df535cdc68b9114e962eb2e3c167c488a61776d877ab6cd04d', 'autonomytax_arxiv2026': '103743b046ccf685e0f3e73e24bdedcb6ef99d11c430e1b8b61f630dd5cdb84a', 'backfire_zenodo2026': '41c37db4c5d353648cc7f8e5979cdd88a480ee964670fc64070ada48570abcf2', 'benchvalid_rs2026': 'd61ded442dcc60058d2a8264b98fbdf35bf1c564cf8248c85a942eed11d56e58', 'bipia_kdd2025': 'bccf303eaa0b2d8b99d9e1613b74c2007d3563b37fe85f79b29e130c5a765894', 'cacheprune_arxiv2025': 'b0eb50127d097fa3562797173ffbeb4e33c73fb422d2053dfe90d0fd1752508e', 'cbc_emnlp2026': 'bb2100ba45b991789a93cad5877fff740bc4d7be6fc49a89edf6ebcb37620669', 'civ_arxiv2025': '0574008075e10a99d67b75c00c065549960a7fafca117d3b650eb61265a31ec7', 'ckptgcg_haips2026': '8b458eb05fe026ae7cfb9b37c0081f8085c299c7b2e3372b93be9c7eff171b1a', 'codel_arxiv2026': '9c33793b5898d7829d32612e0a2844f6f2d1ad1297b591b9fde37938c8b68f0b', 'codt_arxiv2025': 'c46c123da697f283fe6c71dddb7d086601c8611fc1068ee31bcfb92ef1792346', 'coer_arxiv2026': 'e0699817132dc1287baa5d7d212a78671603d75112cbad90d38860668b1b8fb3', 'constsanitlog_icssa2026': 'ec923515a33446a526e3059ba04d8815d8dbb5876ea89fc5b0d2481c3a20544c', 'copa_arxiv2026': 'bf6ba0b49c2f5467b9b204809ee1cbe18ea5f90a8f77222b8b6b63c675e1f7a5', 'countersteer_arxiv2026': '98609e5b33802493830cdc52118f7c7223ab7fe165604ca35677ea8909dc197b', 'crossdt_sbseg2026': '82f7a8a6e9cda3b2a2954893b5b958978cfcb595eb51684aab6185b800f1f628', 'crosslingjudge_ubmk2026': 'e41fc400569a2e221b2c51ad8473d6c68e5c7f120c43d5fbd2db5c03732d8818', 'defensivetokens_aisec2025': '99d07ae16775dfa1e6db09e5421810e70ed24772154ea92759095860b55279d1', 'drip_arxiv2025': '7593569483170cf395b1d1c1faf35b41c2f7cfa1c84eb727a8fe0bc802d1a27b', 'dynppa_arxiv2026': 'ff42e2b6fc33de03ffd137b1633a0d8c2dfb3b8eaebf7387eb795bf126bedf6a', 'edic_emnlp2026': 'a181ddfb9d3adb325e558760ddabde7032ad3cc2ace5c50e2d48dd27420cbefb', 'eroded_zenodo2026': '2ebe9dc89ddaca0f8799ba4afb908a10d9f78515ff34ddddef5f7d199203dacd', 'explosive_arxiv2026': '7bba5a3da908c7c0a07ffdf3742b808c7d3e2280d10aabb1bb687577fe52ceb0', 'fath_arxiv2024': '640de7753db27d8939cf99eaba563643e3bac08590285c56925ecc384ca2e56d', 'gemini_arxiv2025': '264872c281266cdb90bca3265dbf17e7532d74f2d6ddb4b2e823a4e7b2b0ea26', 'glasswall_techrxiv2026': '8f0551ac2a4869b15931d390ad2d4fb9c333b163f7c9bd339f2bebc6d6830c44', 'gptred_arxiv2026': '2df0573618ce5c0fd23619374d44c5a73d16fea4055b1cf25427d4c68dd97e79', 'guardrag_uwthesis2026': 'b94887ab10316131960259ace9d8186767e6f1a0c76bfba3f515060673f84de3', 'gwdpo_arxiv2026': '5a035b2ca79be2ea99d8658c31327a796a60c67d566068ab92decbbd95ac69f6', 'haaulc_rs2026': 'ef99cb4f1a3027712d72f535233b327b37e8a62b8ca323da183de8f9b24f82c6', 'hos_arxiv2026': '5bf48c04b82b616ebcfa258c58bdefde588e23a07b038214a9a81edef9e7d746', 'hybridrag_ijaiml2026': 'da7dff00976ab8de7813df0d421adebe91db02d843d4e7ace46d1852c2bcd0e2', 'icon_arxiv2026': 'd66b6f3f8956177af74dff8ff156303bb42a299cb62570fdc4444894e9031b8f', 'ignorednotrefused_arxiv2026': '66eaeadd07570ea9b42daa2c4cfbbedd5a6fec5b191c38fd19c364d8cf96b35b', 'ih_arxiv2024': '3ccecc3a54bcd28da09c2c019071a933f1cf9d8edea30b51d70214091a090889', 'ihchallenge_arxiv2026': 'cbe6e2d05596c1913b40c40b2d1d75bbe4c7ec702e72b45fbc78170b04e24fbe', 'ihdec_arxiv2026': 'db62db86be1992e45e95a56908f2c8c443a807259e4a8641a2a9d6998abb1a95', 'instrucot_arxiv2026': 'b5e6d1eb6e8899335d81c0ba0e801496c49a3a987f57ca51591051092c236642', 'intentguard_arxiv2025': '731a57851d5f70f557698e0af8b7f9547910ae954bcdf5d10f550b6df14aaa31', 'ipidefrepl_computers2026': '9aa8d154eebc48f890af2140e9ccb4a6f1fc437c6f891ef0820f5a21156dd276', 'ise_iclr2025': '51e42ca8c31710b3c541845086a8512e8f3a4a953c9d011b3cebb0d04c6fa632', 'jatmo_arxiv2023': '12f7a16095d3cb386b09101dfeb552312a2253836bc3aaf390e83e0e44d156c4', 'jatmohouyi_arxiv2025': 'ef0a2a532d56544d4ff97a46d1541b4c1714c17cab52d573ce8ca4362fc72163', 'jia_critical_arxiv2025': 'dd573d4cfa2ebd47573fcb31ede864483d70b95cb962b81af681f3fc975656f7', 'ldd_arxiv2025': '8f671116fb07989196b7cafa98711472ce05d823c8e1ac457c9028b7747d7e5c', 'liu_benchmark_arxiv2023': 'b22bc7f467ff4f4aff7285710ff88106cb100752d323acce10cab647ef0387fe', 'llmailinject_arxiv2025': '9aed9cd1943364a22640cd94bb686f34f04565d23ce5e1fa1d1af220b804b63b', 'llmon_arxiv2026': 'aecf1b8b71407ccb9e17f4910d2e8cd2e4ccef0b435df3019559fc369db74002', 'localalign_ccs2026': '39d987b43917733232ecc1b00763bd6642a6a95ea20e7a13ef1a2060354f939f', 'localrag_applsci2026': 'caa9726a775c95ea6ab104532265f3cb53f83268ba4a54cc4af69e49c17e989d', 'loginject_arxiv2026': 'dec2eba7dc88f457e28793e42652213688620128c56c7f0dc5e06eef9048090d', 'longpibench_arxiv2026': '830c92056b3c015c160ee53fdeb64c5b9a016ee517712ddbf3a11143c176a9bd', 'ltbd_arxiv2026': '06e43d5a0462b6db232c8d2a8e34a319210d39168663018e67ce780bd85adc7f', 'medmas_arxiv2026': 'd6d68b638ff7f13d3aae365e181b9bec53ca097ae5930e07a588b45d11eda0ea', 'metasecalign_arxiv2025': 'a5a104d40f0d701317ee4fd8b985841b919d93cfb8598deee421e5142bdb094b', 'moe_arxiv2025': '13d3985f5aa843e2cc3c69edebeb460b9074492c322c495820eeff7fb9390743', 'msacore_zenodo2026': '1ef9840664aae408696f8f1b7033c5cab2eac83c9c80b2438e1d4dfbc5c4c630', 'municipal_sauc2026': 'e67dd9498b951c3d12c9716da7957c7372d315bcf80421e6cbdfab3f71eab8b7', 'overlays_arxiv2026': '22145d562c982eac23b1f711cc297ae36eb2f4ea32666b9729fd8bed15d3e133', 'peerrevcheck_osf2025': '2ee9ec1af16ce83ee4ae7e918c1721e01fb732daa9373e3cdecebd720b6cecf1', 'pft_arxiv2025': '129b8017dcef3764651ce67a0fbf90f162257499f9cf518b15d56f506125820f', 'piarena_arxiv2026': '739e8a678f189876f8cb4d79948c8d00596e4aee11ad033bf9d185c859711c7e', 'pictionary_arxiv2026': '70c5696d610270902341d921e249529c8db06b43c0b50acabdc13768d133c1c9', 'pidefthesis_upv2026': 'cafc10451b274d513e9371bd25af869771665dbca65b917c9077a7bdf3ac1e80', 'pipette_arxiv2026': 'b63efab1b0051133a3df17cfb0b10e56075100f56c03b8409785b8685f850867', 'pismith_colm2026': '54899cc3cab73929d75d989f2c554f67d8d7edaeaa8b7d9a24aee9c286ce1a04', 'poisonedtools_zenodo2026': 'e8ed86a14a12dad36eb98f20e6780d50bf4526a75c8fbd8d29897509eb1264eb', 'ppa_arxiv2025': 'c3dbf679d514c1a6499f4fcdc6ab452df156badaef8c36ab1760bef028d688ee', 'promptfencing_arxiv2025': '211134ba41d538e852a621666dbe01ba573200a2680ccd4a37e2119360d78c60', 'provtransformer_arxiv2026': '0ff9d7cce32431b0dd001be931c77a3c73f2e3d256612495d9d941e203fcdb85', 'pvst_arxiv2026': 'f3aabf1e92a1765ac5bf5b54bfabf4ce5b1943662cb5451f33669f0183c03fb0', 'ragmitig_ccs2026': '5e795299398aba6e0f9a5e14fdccd99f6317d48d20042dec368e39124a07bea7', 'raised_arxiv2026': '1df62aadc993c9890b99b63bca5a5f71f9206690f995fa9d971a53e33790e05f', 'readable_arxiv2026': '1d7936fef8058207aca5778d5d94cddc57b74a93ebe23cfc7ebd368e04f50aee', 'reasalign_arxiv2026': 'b0081051665dd1940ab853784a64c8dfac771281522217cd5377cba27dde96f4', 'redvisor_arxiv2026': '47f7ccdeade10b9abd2585a8b6c538b5c8560231d89ed9fd133726465861d208', 'reta_arxiv2026': '7991ad1329a3c061b67881a87caa5676c34347e60377e2e5e49e70cc9d64a2b5', 'rlhammer_arxiv2025': '7e785bcb023da892ab42fd8556ccc80e24bab2813bb65e2b6c7727b1596b645a', 'rui_preprints2026': '3bd2727d04f88c9addccaa27ba5cd79032f6325b75fdbd44ff174a8f365ebf85', 'rvr_arxiv2025': '79bf00dc09cd6f70c885e0ddcaf3463051b345a51a456ec1d287edfe7c16b610', 'safereview_arxiv2026': '33afd2eedef3b3bc5122ff7afb90912882ec0d189f12fb3a0d4cbaa08ed148c3', 'secalign_ccs2025': '3e5332ba586ebe425e18d4d6eabf5b87e4a3274c90bbad7efd6e02e585fe8a69', 'secfid_arxiv2026': '091b9e585faefb379664690cde9adea243092b99b2cc9ac201ec1041e44882aa', 'secinfer_arxiv2025': 'd4d31066767e0119a594bb5f89edb9c815a0ae0bd7fd4cc5a1700981a635c899', 'secopd_emnlp2026': 'dc221007c3ce5acb259da2a3d9ff9a1a5a8a6071d4cf6cb5caf0f7d64be194af', 'secpo_arxiv2026': 'bc38c89bb81846f5bc7ce29ef929546414e90870e4267afc8596f077f7c791e0', 'securecai_arxiv2026': 'f79825f465e071dc7759327bea90a2250508a4e8523cf1d9fc8a53b10ca32a84', 'signedprompt_arxiv2024': '75a6ede7a8c0a4ea058ccd92030c010bb709bd9390c436f37d742bf846ac5566', 'smallagentsq_zenodo2026': '1170febc3d3969f06920b55da3d30944a1008b6d14c98937b9954cee10e5801a', 'smalllocal_zenodo2026': '273ebbadbfdf77328619089d91a99fb1c697bb3778ba5ba4f06b13dedbd188bd', 'smallsecure_asiaccs2026': 'c185eb7bf2fd3bbb13302824ae50ef0892195f6e077aaf2f0c1f01a5b8f65211', 'socportab_zenodo2026': 'cde3613fab875687615c083b896ad18b1790960581ea3c0dca4c93034f59bc01', 'spotguard_applsci2026': 'e11147fadd4dcd31c4fe3f95112f2d27dbd674755feacfefdfb5868c0f9e8900', 'spotlighting_arxiv2024': 'f2bd2caf520fed0145230b1ca070e20bb48f4fd68a7ab8c71dbb555dbcc021f0', 'srft_arxiv2026': 'cd6d24ea7d56f016e2a7973d58532b44831cb6ff82537b3302461eabde214f77', 'struq_arxiv2024': 'f1b4e693d85f078f77c0dbfae2596aa44a6a16a6a6cbafe759791099a2e46858', 'surfaceheuristics_arxiv2026': 'ab33917c23790a7bf22e71c5b89e35a1b96ad6e1efad07b656baebb67465e734', 'tab_arxiv2026': '55dc0d1f8631e7d14b150f6d1fbf792e3d82ec57d24587acd0ba3ad1e7f1cd86', 'textpixels_arxiv2026': 'fa9b6f757fdf226b0ad70db42e1270bf1b5d7b96ce0d8449812c1fe5cf6289fe', 'tracer_emnlp2026': '12bdeb33290031fbdb9aec25d9bcf19a91c42356a797da60007ebc777d3249a8', 'transfact_zenodo2026': '09fdf4fa04613e7abd667dd9439dd3e64fb8a9b66f2d056f7256c5314f7eed4a', 'trustmi_arxiv2026': '30432979a4d5c42d3518882ad375618f14170d366f870f3b503617a5b65ce585', 'univdef_arxiv2026': 'c0533f69996cae1568fc0ba107705c1bf35f2d763fcb8a85eb2a7b515d69d85d', 'untrustedcomponent_arxiv2026': '22eb0df80d978539bf25aeb7a871c1ee5c7712cb8f0f49b8fe3bccccb09131d2', 'verih_arxiv2025': 'b61104b66ac74844060f58d9fe94dffc7f3fa216af2c6ddc060b5a98d134751a', 'whosaidthis_zenodo2026': '8abb2e1ed6bae236a4e70c45a383fe9c3a9d6a7dc6bfe6ccc5858c41e8d6784b', 'whosewords_zenodo2026': 'a3a6378398272e1a0df013f56f4a909164ac544f89828ceba750ad463802c374'}
EXPECT_AFTER = {'actconsistency_arxiv2026': '2e3855fffd09b7c4d5d765b0b5bfe17ad856c327563af42bc682b856fb399992', 'adaptiveipi_arxiv2025': 'eb99fdb8740f157aa1d414e02182956b03840700cad3a20f2906daf2d045256a', 'advershield_jtsi2026': 'd12a58035e12285e94fef1819a86d73c601199dfe2cc4c7a141caa24a33abdad', 'advft2022_arxiv2025': '61752fde24964d3a06ad5e497fe69d4778bbf5ee04267a4b9c3f90925a620e59', 'advsim2real_arxiv2026': 'e1d9e6828069c1d2b6ee2d253c967ab1926d53f4e0ca6e82b4ce44fd76353539', 'agentdojo_neurips2024': '579ba1ec91ed348d2fddc0dac159dbaade07d5aba802f7cec740f36e5c4c02e5', 'agentdyn_arxiv2026': 'd0d32b665213b24b8438fe6b7febd0d203c3bf653a6cb04c1747062d8940d0bd', 'agentsafeevo_arxiv2026': '12093f0e15fe405ee067044c71c76748c96c36387661f648ea70d8218f00075c', 'air_arxiv2025': 'e45f5b78f65f13888b7fc3a1a240f0472cbbfc444467b23bb8d100dcf8ecdf29', 'alwaysfall_arxiv2026': 'aa78b1618af7c62dd39a6063e89237c6aa275b7139738d224b5d413731a97771', 'aort_etrij2026': '1da167f5de0dcdfa3af3e1ad085c3e58c757d2729ae4431c3ca2128db44791b2', 'arlas_arxiv2025': '65f0443c9a7e26f2b945ea3657d4717e0b9b29c6c3f28fb59c4f9e182e3d086a', 'asb_iclr2025': '4f6928bb23c80c7b1aab49423d261c37bbcdd78975d4290d6c6e99831c8bb741', 'aside_iclr2026': '1e8b5e8480a51239e79e03e7b39400d7e660897795cb0669be8a39b8ead6cad4', 'astra_arxiv2025': 'a21a4a3cf811be3cf5fa7f8b46992cabb1ac30c8f160bf4a854a75c584e4a8e3', 'attackdefence_arxiv2024': '848f324d0cf3d586fc2a4fb812d8287c793a138d87601cc9ba1c552e7cb3334e', 'attackermovessecond_arxiv2025': 'c315d262660a2caa982fc9af3d38270a9d689b6363e4db02a8e61855861b6c20', 'autodojo_arxiv2026': '036ab20f7f1c91510dfffcd4e870cf2daf19d998a619597be8614f9003ca13e3', 'autonomytax_arxiv2026': 'f5d21d5f79278b5c928b9161e908badb823b73c983422a256d29eb62216aa822', 'backfire_zenodo2026': '0a40d937f8ff6ef76460473a578a6a0b1aa09114f0f5b0ee54ae026d079fe601', 'benchvalid_rs2026': 'e2dcfda9bbba8f63a24e9cf88bddae9ca27647b0f2282fe8bcc33ffc0284d21a', 'bipia_kdd2025': 'bccf303eaa0b2d8b99d9e1613b74c2007d3563b37fe85f79b29e130c5a765894', 'cacheprune_arxiv2025': '993d7af61e852f1b0665cfaf348a2a4468b141c8a1d6a6ab73723986d619014f', 'cbc_emnlp2026': 'bb2100ba45b991789a93cad5877fff740bc4d7be6fc49a89edf6ebcb37620669', 'civ_arxiv2025': '455b1bfa1673a57901031e46a1cbfcf00e057deff67223293397bf9ac41cfcf3', 'ckptgcg_haips2026': '8b458eb05fe026ae7cfb9b37c0081f8085c299c7b2e3372b93be9c7eff171b1a', 'codel_arxiv2026': 'fa6b87b555a1bee088a0424896255d57103b5de27e774ae3ebf3e416898a2dd5', 'codt_arxiv2025': '4076c6370904f4b6e11ac59efa74ce3af858511c67b0ddfad57dc43aa83765f3', 'coer_arxiv2026': 'e0699817132dc1287baa5d7d212a78671603d75112cbad90d38860668b1b8fb3', 'constsanitlog_icssa2026': '9b0d02aa629cdc4a15b921686d9b18bd4dab5d53bef7a5bacb165de84e98d642', 'copa_arxiv2026': '0c04eaf66bfeb39920d6ad92a7dfe321da8b9046580a417e3079580058454c81', 'countersteer_arxiv2026': '71fec72896123e9c80c92c2e90f667565dd5145c52f851a58b8473fa5270181a', 'crossdt_sbseg2026': '0414b5b05804c76071f4c598aca35a1ceeebeda7b0b51dd6689f65a15f4ebab6', 'crosslingjudge_ubmk2026': 'e41fc400569a2e221b2c51ad8473d6c68e5c7f120c43d5fbd2db5c03732d8818', 'defensivetokens_aisec2025': '99d07ae16775dfa1e6db09e5421810e70ed24772154ea92759095860b55279d1', 'drip_arxiv2025': '7593569483170cf395b1d1c1faf35b41c2f7cfa1c84eb727a8fe0bc802d1a27b', 'dynppa_arxiv2026': '91fb9f7f318071bb6af6b40c6cf5d2e32cecf9c93d8e597ee1d6a21e9018f856', 'edic_emnlp2026': 'a181ddfb9d3adb325e558760ddabde7032ad3cc2ace5c50e2d48dd27420cbefb', 'eroded_zenodo2026': '89124f39a3305edfeacda533fad1d6e3d7e4e8880666548518cc0a7774fe29dd', 'explosive_arxiv2026': '3518a6ceaf71dd9163d86de0ebf8bfaf7cb3860b695bf73a7a7516a5ef26722a', 'fath_arxiv2024': '640de7753db27d8939cf99eaba563643e3bac08590285c56925ecc384ca2e56d', 'gemini_arxiv2025': '4abde41db908bbc18ff9cafe79b80d3dfb8756eee9e9038b8edbfcf1e9846b41', 'glasswall_techrxiv2026': '9acfeb4cf82aaa656b9f9383ee6de7e863e8b92234a837a604bc0dbf41b1ad3a', 'gptred_arxiv2026': 'cf874a3ff62375656e912036270e9da15f143831909bc3d5e36ebf6c4f3658f7', 'guardrag_uwthesis2026': 'b94887ab10316131960259ace9d8186767e6f1a0c76bfba3f515060673f84de3', 'gwdpo_arxiv2026': '5a035b2ca79be2ea99d8658c31327a796a60c67d566068ab92decbbd95ac69f6', 'haaulc_rs2026': '7d602332f5161ef0930699bece5244dbd94c7b252e3429053c69ead30c7cd49e', 'hos_arxiv2026': '5bf48c04b82b616ebcfa258c58bdefde588e23a07b038214a9a81edef9e7d746', 'hybridrag_ijaiml2026': '5aecee06596dde40295a047f2e5efdd69e9201a12613937926aa74de37abfdd0', 'icon_arxiv2026': '7ce1b963ca8aed645bddf74dc6f6677beedb335e36c3f68960341070816e855f', 'ignorednotrefused_arxiv2026': 'ad83ec3471bbafbd7c82e2794e9f9379885fc304a8c9105543e07e36805718cd', 'ih_arxiv2024': '3ccecc3a54bcd28da09c2c019071a933f1cf9d8edea30b51d70214091a090889', 'ihchallenge_arxiv2026': 'cbe6e2d05596c1913b40c40b2d1d75bbe4c7ec702e72b45fbc78170b04e24fbe', 'ihdec_arxiv2026': '223b2e7e243621c4d8b6c1f186c0c2235be37d21e541e32b4c07e9ea1df2c06f', 'instrucot_arxiv2026': '4a0fc04069320161b1f522e35252abafa9841caeb06520c47935551e89353566', 'intentguard_arxiv2025': '6b0431daad21d560f0e7851b835876f71c3c016459641d2d3d32c6ddbb18fa05', 'ipidefrepl_computers2026': '0d6bb8b75a7295da11cfad9486f7bae048a36b190c7095a14b8a50788d3c2ab2', 'ise_iclr2025': '20d809a9a3c9bddcf191a0f197d9cff7fd100becc951b1eb3bc60d1f0acbdb59', 'jatmo_arxiv2023': '12f7a16095d3cb386b09101dfeb552312a2253836bc3aaf390e83e0e44d156c4', 'jatmohouyi_arxiv2025': 'f3d44705d87c1729316912466124d5d40e28643ac0e777bd49f250948c7dcf27', 'jia_critical_arxiv2025': 'dd573d4cfa2ebd47573fcb31ede864483d70b95cb962b81af681f3fc975656f7', 'ldd_arxiv2025': 'fb8413cb024f7a0d13934f4b20df44acfe769cc26fa6b904575b3c7d0c32ba66', 'liu_benchmark_arxiv2023': '1071f911117e930664e6adb4860b9de6d61cff6ba2a333c6a4ffe88ec7eea65e', 'llmailinject_arxiv2025': '9aed9cd1943364a22640cd94bb686f34f04565d23ce5e1fa1d1af220b804b63b', 'llmon_arxiv2026': 'aecf1b8b71407ccb9e17f4910d2e8cd2e4ccef0b435df3019559fc369db74002', 'localalign_ccs2026': '39d987b43917733232ecc1b00763bd6642a6a95ea20e7a13ef1a2060354f939f', 'localrag_applsci2026': '70dea8fb0ba256e476f9fb1fdb87a6ea78a4c70d38202f01f026f94c92f3c820', 'loginject_arxiv2026': '6a46b1571819fd1f13a90ac1cb5e7d366812cd552d029f1244558fc431d91c34', 'longpibench_arxiv2026': 'f4db1779bd7ad7421e6cb1632e77224f4a744c6f84106a0e50736ca792380969', 'ltbd_arxiv2026': '06e43d5a0462b6db232c8d2a8e34a319210d39168663018e67ce780bd85adc7f', 'medmas_arxiv2026': 'a17f2a040712f8eb11736f1280b069d8a90433b55b2ff7a574cd6cf7a9fa3ea7', 'metasecalign_arxiv2025': 'a5a104d40f0d701317ee4fd8b985841b919d93cfb8598deee421e5142bdb094b', 'moe_arxiv2025': '13d3985f5aa843e2cc3c69edebeb460b9074492c322c495820eeff7fb9390743', 'msacore_zenodo2026': '1ef9840664aae408696f8f1b7033c5cab2eac83c9c80b2438e1d4dfbc5c4c630', 'municipal_sauc2026': 'faa0a40c6ca65d42a3abf82846deeaf77054f0501986d893e70608b0e52268f9', 'overlays_arxiv2026': '22145d562c982eac23b1f711cc297ae36eb2f4ea32666b9729fd8bed15d3e133', 'peerrevcheck_osf2025': 'c72917fc3cc4b7ea8e006f508a075412df0dae872758fa60608fa0beb4dd0890', 'pft_arxiv2025': '129b8017dcef3764651ce67a0fbf90f162257499f9cf518b15d56f506125820f', 'piarena_arxiv2026': '739e8a678f189876f8cb4d79948c8d00596e4aee11ad033bf9d185c859711c7e', 'pictionary_arxiv2026': '0cdec7e9fabd0e869ff8dcaa294e902521196743ee02945d8c5c4f99b5579150', 'pidefthesis_upv2026': 'c6e29c0a110910db2f14ed76022ab3f58448709f39d3018975160bf7de348953', 'pipette_arxiv2026': '423c22ed0292693efeff44137e038e4ba70af47925b588fcc0236a2875896b26', 'pismith_colm2026': '7824aa0de332d3c3013a322837e72bbd59c851f09a5e3e7745b5f4226c8f52c7', 'poisonedtools_zenodo2026': 'c4c03527042b46d1c749d7d37dc4a2c83ba98c7f1342db45e55bd15ce1fa2740', 'ppa_arxiv2025': '8604b476c8ba2f6068a71fe4525b0916429490f71dc864e9feffb86744252200', 'promptfencing_arxiv2025': '211134ba41d538e852a621666dbe01ba573200a2680ccd4a37e2119360d78c60', 'provtransformer_arxiv2026': '3ddbd2f8e71c0ad7da449b7d5bf80ef406a1b90fde60d175a0569088ebc8f4ff', 'pvst_arxiv2026': 'fef4b575532bccbff37eeb1ea5aeb71c73db6c59cee2b1b0c93c7c9eacaa7242', 'ragmitig_ccs2026': '1ef72fae5d0a67b8c45e766c9df47b3ceb495e6a5d74a80ac0cb98fb4baa0079', 'raised_arxiv2026': '1df62aadc993c9890b99b63bca5a5f71f9206690f995fa9d971a53e33790e05f', 'readable_arxiv2026': 'a551384075d1fcf308410f5fa6adbc71200e3abdbe38e0b14e51a33fdd60e07a', 'reasalign_arxiv2026': 'b0081051665dd1940ab853784a64c8dfac771281522217cd5377cba27dde96f4', 'redvisor_arxiv2026': '47f7ccdeade10b9abd2585a8b6c538b5c8560231d89ed9fd133726465861d208', 'reta_arxiv2026': 'd83f35ac620dffbbdc13f0e0b95e2c18e91999f629d928117aa06db900219585', 'rlhammer_arxiv2025': '7e785bcb023da892ab42fd8556ccc80e24bab2813bb65e2b6c7727b1596b645a', 'rui_preprints2026': 'ad6f5e45e150061f8ff07bc72240e32ed50215ac89920ecdbfa287981f2ebf65', 'rvr_arxiv2025': '877bd84e290e065c5cceef7f5dd8816cc852ec64c6e6fd58a3fb01455eaa77ab', 'safereview_arxiv2026': 'ec8e4fa6ee6ce1e2b1353e7946a43f8fd3c1a8034ffffa4687020a574f497afb', 'secalign_ccs2025': '3e5332ba586ebe425e18d4d6eabf5b87e4a3274c90bbad7efd6e02e585fe8a69', 'secfid_arxiv2026': '002011b9133f19197f42ee51ee37d30844127ed20fecc7c0a4afb4ce5a87eea8', 'secinfer_arxiv2025': 'c7ed0160346d00dea6d105c4084e895a872db28e0ad5f1d4924a2f12aa6e8746', 'secopd_emnlp2026': '8e222e7404f853a6a56a51d20ad6d3b26df61c36e29dff7d9bd3cca6ef210447', 'secpo_arxiv2026': 'bc38c89bb81846f5bc7ce29ef929546414e90870e4267afc8596f077f7c791e0', 'securecai_arxiv2026': '5d7d559a60ba6186a60de08e78fab122ff0761cf7f45475072864b10870735f8', 'signedprompt_arxiv2024': '75a6ede7a8c0a4ea058ccd92030c010bb709bd9390c436f37d742bf846ac5566', 'smallagentsq_zenodo2026': '23995d40dacb4a5301539a9b57c544473187a4af88f60018496ac2c273d23866', 'smalllocal_zenodo2026': 'c07d825785b612b0cf89cb160d67d063cb64b82be394226525117e64730829c5', 'smallsecure_asiaccs2026': 'c185eb7bf2fd3bbb13302824ae50ef0892195f6e077aaf2f0c1f01a5b8f65211', 'socportab_zenodo2026': '2c8ba54b01f098c45d8f9ee07f7dcb8792d24769f90aed265ef2f2a08203797a', 'spotguard_applsci2026': '5b50170d21079f86b8856fb61ffa148a44be504c80dcff2ae58edcdbe7541394', 'spotlighting_arxiv2024': '32d09dbeb197c5375c55cbe6de20ffb42804b53db0fe47cb2505612e557fe49b', 'srft_arxiv2026': 'cd6d24ea7d56f016e2a7973d58532b44831cb6ff82537b3302461eabde214f77', 'struq_arxiv2024': 'f1b4e693d85f078f77c0dbfae2596aa44a6a16a6a6cbafe759791099a2e46858', 'surfaceheuristics_arxiv2026': 'ab33917c23790a7bf22e71c5b89e35a1b96ad6e1efad07b656baebb67465e734', 'tab_arxiv2026': '5544dfdf49dba7f7174cb5454e7e158ac00a61dcbf79b8270c038031be77d7b0', 'textpixels_arxiv2026': 'c9a69eb481f3e42358989453af4f2c93cf2276a84d9f2853cb6d45382432492f', 'tracer_emnlp2026': 'da094a046f7c6e8455b9f7729ec41842bd1e4f2834c8eb9c7832b423770e2892', 'transfact_zenodo2026': '6d631502ddce444dbad5ce75c2fd2cbd9c1bbeab9d60e1249f80dabacd95a993', 'trustmi_arxiv2026': '30432979a4d5c42d3518882ad375618f14170d366f870f3b503617a5b65ce585', 'univdef_arxiv2026': 'f04f984806d64d7a5b18b837b3cad9be8ff2a21b1c3111e0564841aef3a405c8', 'untrustedcomponent_arxiv2026': '9cca2517a9c1b1de68c2904c351a821b1c912225560cd10d37599490857710fb', 'verih_arxiv2025': '774f1dbc1c3ad016657b25a80524fe7274c171fb3c83637cb3ea229332cb0638', 'whosaidthis_zenodo2026': 'f2873387cf56865d8eddd41066acf21db0d425748d3b24d4f1abd59d27ca6c74', 'whosewords_zenodo2026': '3074ae04c6851c9d0cea85d312d8ddb9950823656579cc9508baeb4761132006'}
def sha(t):
    return hashlib.sha256(t.encode("utf-8")).hexdigest()
texts = {f.stem: f.read_text(encoding="utf-8") for f in sorted(paths.RECORDS.glob("*.yaml"))}
off = sorted(set(texts) ^ set(EXPECT_BEFORE)) + [r for r in EXPECT_BEFORE if r in texts and sha(texts[r]) != EXPECT_BEFORE[r]]
if off:
    raise SystemExit(f"STOP: the record files are not the ones the re-code was built from: {off[:10]}")
recs = {rid: yaml.safe_load(t) for rid, t in texts.items()}
before = copy.deepcopy(recs)
rules_before = [f"{rid}: {p}" for rid, r in recs.items() for p in extract.rule_problems(r)]
changes = search.read_csv(paths.REPORTS / "recode_04r_changes.csv")
probs = extract.apply_changes(recs, changes)
for p in probs:
    print("PROBLEM:", p)
if probs:
    raise SystemExit("STOP: the change list does not fit the records on file; nothing was written")
after = {rid: yaml.safe_dump(r, sort_keys=False, width=220, allow_unicode=True) for rid, r in recs.items()}
bad = [rid for rid in after if sha(after[rid]) != EXPECT_AFTER[rid]]
rules_after = [f"{rid}: {p}" for rid, r in recs.items() for p in extract.rule_problems(r)]
for p in rules_after:
    print("RULE:", p)
if bad or rules_after:
    raise SystemExit(f"STOP: re-coded files differ from the expected digests ({bad[:5]}) or break the amended rules; nothing was written")
written = 0
for rid, t in after.items():
    if t != texts[rid]:
        (paths.RECORDS / f"{rid}.yaml").write_text(t, encoding="utf-8")
        written += 1
print(f"amended-rule problems before the re-code: {len(rules_before)}; after: {len(rules_after)}")
print(f"changed fields: {len(changes)}; record files rewritten: {written} of {len(after)}")
print("by field:", dict(Counter(r["field"].split(".")[0] for r in changes)))
for f in ("axis1", "isolation", "signal_dependence"):
    print(f"  {f}:", dict(Counter(f"{r['old']} -> {r['new']}" for r in changes if r["field"] == f).most_common()))

# every field re-checked: three defence fields in every defence record, four coverage fields in every report, and any other changed field
chg = {(r["report"], r["unit"], r["field"]): r for r in changes}
full, seen = [], set()
def row(rid, unit, field, old_rec):
    k = (rid, unit, field); seen.add(k); c = chg.get(k)
    full.append({"report": rid, "unit": unit, "field": field, "old": extract.encode(extract.field_value(old_rec, unit, field)),
                 "new": extract.encode(extract.field_value(recs[rid], unit, field)), "changed": "yes" if c else "no",
                 "reason": c["reason"] if c else "checked under Amendment 2; no change", "page": c["page"] if c else ""})
for rid in sorted(recs):
    for d in before[rid]["defences"]:
        for f in ("axis1", "isolation", "signal_dependence"):
            row(rid, "defences:" + d["id"], f, before[rid])
    for q in ("R1", "R3ns", "R3cp", "R3safe"):
        row(rid, "coverage", q, before[rid])
for r in changes:
    if (r["report"], r["unit"], r["field"]) not in seen:
        row(r["report"], r["unit"], r["field"], before[r["report"]])
search.write_csv(paths.REPORTS / "recode_04r.csv", full, ["report", "unit", "field", "old", "new", "changed", "reason", "page"])
n_def = sum(len(r["defences"]) for r in recs.values())
print(f"reports/recode_04r.csv: {len(full)} fields re-checked ({n_def} defence records x 3, {len(recs)} reports x 4, "
      f"{len(full) - 3 * n_def - 4 * len(recs)} other changed fields); {sum(r['changed'] == 'yes' for r in full)} changed")

## Stage 4. Revalidate every record against the frozen schema and the corpus

In [ ]:
import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
problems = []
for rid in sorted(recs):
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    p, _ = schema.validate_record(rec, S1)
    problems += [f"{rid}: {x}" for x in p]
man = search.read_csv(paths.REPORTS / "defences.csv")
included = sorted(m["slug"] for m in man if m.get("decision") == "include")
pages = {}
for rid, r in recs.items():
    pdf = paths.DATA_DEFENCES / f"{r['report']['slug']}.pdf"
    if pdf.exists():
        pages[rid] = len(PdfReader(str(pdf)).pages)
    else:
        problems.append(f"{rid}: {pdf.name} not in data/defences")
problems += extract.cross_record_problems(recs, included, pages)
for p in problems:
    print("PROBLEM:", p)
if problems:
    raise SystemExit("STOP: re-coded records do not validate")
print(f"{len(recs)} records validate against the frozen schema; one record per included study ({len(included)}); "
      f"every page within its PDF ({len(pages)} PDFs read for page counts)")

## Stage 5. Every second-pass disagreement, resolved

One row per disagreement in `reports/second_pass_disagreements_04.csv`: the first and second codes, the value after this notebook, which side it matches, and why. Fields below the threshold take the re-coded value; fields that met the threshold were checked one disagreement at a time. The agreement figures in `reports/agreement_04.csv` are the figures before resolution and are not changed.

In [ ]:
%%writefile reports/second_pass_resolution_04r.csv
report,unit,field,first,second,final,resolved_to,reason
actconsistency_arxiv2026,coverage,R2attack,not_reported,direct,not_reported,first,"kept: CleanTask_any is task done and attack failed, coded R3safe as the corpus codes this joint metric (codel UA, coer Safe-U); no utility-under-attack value is printed"
actconsistency_arxiv2026,coverage,R3safe,direct,not_reported,direct,first,"kept: CleanTask_any (task done and attack failed, p.17) is the joint R3safe metric, as coded for codel and coer"
actconsistency_arxiv2026,act_q17,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,act_q17,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,act_q17,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,act_q8,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,act_q8,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,act_q8,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,act_phi,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,act_phi,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,act_phi,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,act_oss,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,act_oss,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,act_oss,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,act_gem,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,act_gem,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,act_gem,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,bct_q17,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,bct_q17,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,bct_q17,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,bct_q8,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,bct_q8,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,bct_q8,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,bct_phi,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,bct_phi,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,bct_phi,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,bct_oss,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,bct_oss,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,bct_oss,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,bct_gem,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
actconsistency_arxiv2026,bct_gem,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
actconsistency_arxiv2026,bct_gem,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
actconsistency_arxiv2026,secalign_q8,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
actconsistency_arxiv2026,secalign_q8,axis1,no_signal,chat_roles,delimiters_markers,neither,the report describes SecAlign as extending StruQ's structured queries that delimit instructions from data; no_signal would say SecAlign shows no source (p.3)
advershield_jtsi2026,coverage,R2clean,not_reported,direct,not_reported,first,kept: no clean utility is printed for TAGD alone
advershield_jtsi2026,coverage,R2attack,not_reported,unclear,not_reported,first,kept: no utility under attack is printed for TAGD alone
advershield_jtsi2026,tagd,axis1,internal_intervention,no_signal,no_signal,second,"TAGD scales its noise by attention salience, not by the source of the token; no source is shown at inference (p.3)"
advershield_jtsi2026,second pass,defence set,(not coded),AdverShield-LLM without PLSA (TAGD + IACRS),(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
advsim2real_arxiv2026,coverage,R1,not_reported,proxy,not_reported,first,kept: no attack success rate is printed; the adversary's success flips are a training reward
advsim2real_arxiv2026,cap3,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
advsim2real_arxiv2026,cap3,axis2,preference_optimisation,adversarial_rl,preference_optimisation,first,"kept: Stage 1 trains with GRPO and no injections, so no adversary is in the loop; RL without an adversary in the loop is coded preference_optimisation across the corpus"
agentsafeevo_arxiv2026,nolw,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
agentsafeevo_arxiv2026,nodw,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
agentsafeevo_arxiv2026,noaw,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
agentsafeevo_arxiv2026,static,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
agentsafeevo_arxiv2026,static,axis2,preference_optimisation,adversarial_rl,preference_optimisation,first,kept: the Static-AT attacker is fixed and is not trained against the defender; RL without an adaptive adversary is coded preference_optimisation across the corpus
agentsafeevo_arxiv2026,spot,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
agentsafeevo_arxiv2026,spot,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.8)"
autodojo_arxiv2026,rem_g4m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,rem_g54m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,rem_gem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,rem_ds,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,rem_hk,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,spot_g4m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,spot_g54m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,spot_gem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,spot_ds,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
autodojo_arxiv2026,spot_hk,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
cacheprune_arxiv2025,second pass,defence set,(not coded),"CachePrune ablation variants (mask transfer, k, attribution loss, alpha, without phi, N)",(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
cacheprune_arxiv2025,second pass,defence set,(not coded),Delimiting,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
cacheprune_arxiv2025,second pass,defence set,(not coded),Datamarking,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
cacheprune_arxiv2025,second pass,defence set,(not coded),Sandwich,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
cacheprune_arxiv2025,second pass,defence set,(not coded),Encode_Base64,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
civ_arxiv2025,coverage,R2clean,not_reported,proxy,not_reported,first,kept: Sim compares output tokens with the undefended model's output; it does not score the user's task and stays general_utility
civ_arxiv2025,civ_l3,signal_dependence,measured,not_reported,baseline_only,neither,"the Table 3 variants change how the trust labels are signed, propagated or cached; none runs CIV without the labels; the signal-removed condition is the frozen undefended model, which is reported (p.9)"
civ_arxiv2025,civ_unsigned_l3,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
civ_arxiv2025,civ_unsigned_l3,signal_dependence,not_applicable,not_reported,baseline_only,neither,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.9)"
codt_arxiv2025,codt_0,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_0,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_1,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_1,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_2,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_2,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_3,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_3,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_4,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_4,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_5,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_5,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_6,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_6,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_7,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_7,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_8,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_8,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_9,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_9,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_10,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_10,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_11,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_11,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_12,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_12,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_13,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_13,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_14,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_14,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_15,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_15,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_16,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_16,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
codt_arxiv2025,codt_17,isolation,not_applicable,no_ablation,no_ablation,second,a defence this report runs as a method; no run with a component removed or changed is reported (whole text searched)
codt_arxiv2025,codt_17,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
copa_arxiv2026,copa_l31,axis2,preference_optimisation,adversarial_rl,preference_optimisation,first,kept: COPA is continual preference optimisation over a fixed stream of attack variants
copa_arxiv2026,secalign_l31,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
copa_arxiv2026,secalign_l31,axis1,no_signal,delimiters_markers,delimiters_markers,second,the report does not describe SecAlign's layout; coded as the other unspecified re-runs of SecAlign in the corpus are coded; no_signal would say SecAlign shows no source
copa_arxiv2026,second pass,defence set,(not coded),COPA ablation variants,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
explosive_arxiv2026,msa70,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
explosive_arxiv2026,epdpo,isolation,not_applicable,no_ablation,no_ablation,second,"a defence this report runs as a method; no run with a component removed or changed is reported (the reported steps are checkpoints, not components)"
intentguard_arxiv2025,sep_qwen,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
intentguard_arxiv2025,sep_qwen,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.7)"
intentguard_arxiv2025,sep_oss,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
intentguard_arxiv2025,sep_oss,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.7)"
intentguard_arxiv2025,second pass,defence set,(not coded),IntentGuard ablation variants,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
ipidefrepl_computers2026,spot_g54,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ipidefrepl_computers2026,spot_g54,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.19)"
ipidefrepl_computers2026,spot_g54m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ipidefrepl_computers2026,spot_g54m,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.19)"
ipidefrepl_computers2026,spot_cs46,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ipidefrepl_computers2026,spot_cs46,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.19)"
ipidefrepl_computers2026,rep_g54,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ipidefrepl_computers2026,rep_g54m,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ipidefrepl_computers2026,rep_cs46,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
jia_critical_arxiv2025,struq_llama3_8bi_rel,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
jia_critical_arxiv2025,secalign_llama3_8bi_rel,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
jia_critical_arxiv2025,secalign_llama3_8bi_rel,axis1,chat_roles,delimiters_markers,chat_roles,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
jia_critical_arxiv2025,second pass,defence set,(not coded),Instruction Hierarchy (as deployed in GPT-4o-mini),(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
loginject_arxiv2026,report,role,mixed,benchmark,benchmark,second,the report's contribution is measurement on its log-injection benchmark and an evaluation of existing defence layers (p.19); it proposes no defence of its own (p.19)
loginject_arxiv2026,spot,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
loginject_arxiv2026,spot,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.15)"
ltbd_arxiv2026,ltbd_l3,signal_dependence,measured_other_checkpoint,measured,measured_other_checkpoint,first,rule 9: measured on another checkpoint of the method
ltbd_arxiv2026,ltbd_l3,axis2,soft_tokens_frozen;preference_optimisation,soft_tokens_frozen,soft_tokens_frozen;preference_optimisation,first,kept: the LTBD loss has a DPO-style term against the response on the attacked input
ltbd_arxiv2026,ltbd_l31,signal_dependence,measured_other_checkpoint,measured,measured_other_checkpoint,first,rule 9: measured on another checkpoint of the method
ltbd_arxiv2026,ltbd_l31,axis2,soft_tokens_frozen;preference_optimisation,soft_tokens_frozen,soft_tokens_frozen;preference_optimisation,first,kept: the LTBD loss has a DPO-style term against the response on the attacked input
ltbd_arxiv2026,ltbd_f3,signal_dependence,measured_other_checkpoint,measured,measured_other_checkpoint,first,rule 9: measured on another checkpoint of the method
ltbd_arxiv2026,ltbd_f3,axis2,soft_tokens_frozen;preference_optimisation,soft_tokens_frozen,soft_tokens_frozen;preference_optimisation,first,kept: the LTBD loss has a DPO-style term against the response on the attacked input
ltbd_arxiv2026,ltbd_q25,signal_dependence,measured_other_checkpoint,measured,measured_other_checkpoint,first,rule 9: measured on another checkpoint of the method
ltbd_arxiv2026,ltbd_q25,axis2,soft_tokens_frozen;preference_optimisation,soft_tokens_frozen,soft_tokens_frozen;preference_optimisation,first,kept: the LTBD loss has a DPO-style term against the response on the attacked input
ltbd_arxiv2026,pre_l3,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,pre_l3,signal_dependence,not_applicable,measured,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
ltbd_arxiv2026,pre_l3,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,bce_l3,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,bce_l3,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
ltbd_arxiv2026,pre_l31,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,pre_l31,signal_dependence,not_applicable,measured,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
ltbd_arxiv2026,pre_l31,axis1,no_signal,delimiters_markers,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,bce_l31,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,bce_l31,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
ltbd_arxiv2026,dt_l3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,dt_l3,axis1,chat_roles,delimiters_markers,chat_roles,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,dt_l31,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,dt_l31,axis1,chat_roles,delimiters_markers,chat_roles,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,dt_f3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,dt_f3,axis1,chat_roles,delimiters_markers,chat_roles,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,dt_q25,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,dt_q25,axis1,chat_roles,delimiters_markers,chat_roles,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
ltbd_arxiv2026,struq_l3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,struq_l31,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,struq_f3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,struq_q25,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,sec_l3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,sec_l31,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,sec_f3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,sec_q25,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
ltbd_arxiv2026,second pass,defence set,(not coded),TextGrad (optimised defensive prompt),(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
ltbd_arxiv2026,second pass,defence set,(not coded),Reminder,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
ltbd_arxiv2026,second pass,defence set,(not coded),Sandwich,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
municipal_sauc2026,coverage,R1,direct,aggregate_only,aggregate_only,second,every result is pooled over the models (the defences' base is a pooled set); aggregate_only applies whatever the instrument (p.7)
municipal_sauc2026,coverage,R2clean,direct,aggregate_only,aggregate_only,second,follows from the same reading as R1: pooled over the models (p.7)
municipal_sauc2026,dm,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
municipal_sauc2026,dm,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.9)"
municipal_sauc2026,sq,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
municipal_sauc2026,sq,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.9)"
pictionary_arxiv2026,coverage,R3ns,not_reported,proxy,not_reported,first,kept: no result in the extraction scope scores whether rendered content is ignored
pictionary_arxiv2026,pic_opus,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_opus,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_haiku,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_haiku,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_g55,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_g55,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_nano,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_nano,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_mini,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_mini,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_qwen,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_qwen,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_kimi,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_kimi,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_grok,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_grok,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_gpro,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_gpro,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_glite,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_glite,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.6)"
pictionary_arxiv2026,pic_g54,isolation,not_applicable,ablation_present,ablation_present,second,"Appendix B runs Pictionary with the rendering changed (pagination, code rendering, alternative visual styles) (p.19)"
pictionary_arxiv2026,pic_g54,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.7)"
pictionary_arxiv2026,aud_gaud,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
pictionary_arxiv2026,aud_gaud,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.8)"
pictionary_arxiv2026,aud_qomni,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
pictionary_arxiv2026,aud_qomni,signal_dependence,not_applicable,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.8)"
pismith_colm2026,msa8b,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
pismith_colm2026,sand_q,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
pismith_colm2026,instr_q,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
pismith_colm2026,second pass,defence set,(not coded),Instruction Hierarchy (GPT-4o-mini),(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
ppa_arxiv2025,ppa_g35,isolation,not_applicable,ablation_present,ablation_present,second,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported (p.5)"
ppa_arxiv2025,ppa_g35,axis2,prompting;fixed_inference_rule,prompting,prompting;fixed_inference_rule,first,kept: drawing random separators and assembling the prompt is a fixed inference-time rule
ppa_arxiv2025,ppa_g4,isolation,not_applicable,ablation_present,ablation_present,second,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported (p.5)"
ppa_arxiv2025,ppa_g4,axis2,prompting;fixed_inference_rule,prompting,prompting;fixed_inference_rule,first,kept: drawing random separators and assembling the prompt is a fixed inference-time rule
ppa_arxiv2025,ppa_l3,isolation,not_applicable,ablation_present,ablation_present,second,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported (p.5)"
ppa_arxiv2025,ppa_l3,axis2,prompting;fixed_inference_rule,prompting,prompting;fixed_inference_rule,first,kept: drawing random separators and assembling the prompt is a fixed inference-time rule
ppa_arxiv2025,ppa_ds,isolation,not_applicable,ablation_present,ablation_present,second,"the system-prompt format is changed (PRE, ESD, EIBD, RIZD, WBR) and each is reported (p.5)"
ppa_arxiv2025,ppa_ds,axis2,prompting;fixed_inference_rule,prompting,prompting;fixed_inference_rule,first,kept: drawing random separators and assembling the prompt is a fixed inference-time rule
ppa_arxiv2025,second pass,defence set,(not coded),"PPA system-prompt styles (PRE, ESD, EIBD, RIZD, WBR)",(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
raised_arxiv2026,raised_gem,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
raised_arxiv2026,raised_gem,trains_carry_through,yes_exact_targets,no,yes_exact_targets,first,"kept: the teacher targets act on legitimate guidance carried in tool outputs (p.2), so training rewards using instruction-bearing data"
raised_arxiv2026,raised_gem,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
raised_arxiv2026,raised_nem,signal_dependence,not_applicable,not_reported,not_applicable,first,rule 9 (Amendment 2): axis 1 is no_signal
raised_arxiv2026,raised_nem,trains_carry_through,yes_exact_targets,no,yes_exact_targets,first,"kept: the teacher targets act on legitimate guidance carried in tool outputs (p.2), so training rewards using instruction-bearing data"
raised_arxiv2026,raised_nem,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
raised_arxiv2026,sapp_gem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,sapp_nem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,sand_gem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,sand_nem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,spot_gem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,spot_nem,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
raised_arxiv2026,second pass,defence set,(not coded),RAISED ablation variants (data and objective),(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
rvr_arxiv2025,struq_l3,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
rvr_arxiv2025,struq_q2,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
rvr_arxiv2025,struq_l31,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
rvr_arxiv2025,second pass,defence set,(not coded),"RVR variants (split only, dynamic tags, without in-context examples)",(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
rvr_arxiv2025,second pass,defence set,(not coded),Sandwich,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
rvr_arxiv2025,second pass,defence set,(not coded),Reminder,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
rvr_arxiv2025,second pass,defence set,(not coded),Instructional,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
rvr_arxiv2025,second pass,defence set,(not coded),Spotlight,(not coded),first,"kept as coded: the entry lies outside the first coding's extraction scope, which names what was coded"
securecai_arxiv2026,secai_nosan,signal_dependence,not_reported,not_applicable,not_applicable,second,axis 1 is no_signal: no inference-time source signal
securecai_arxiv2026,cai,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
securecai_arxiv2026,cai,axis2,sft,sft;preference_optimisation,sft;preference_optimisation,second,the report describes Constitutional AI as supervised critique-revision followed by RL from AI feedback (p.1); the RL stage is preference optimisation (p.1)
securecai_arxiv2026,ih,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
securecai_arxiv2026,ih,signal_dependence,baseline_only,not_reported,not_applicable,neither,axis 1 is no_signal: no inference-time source signal
securecai_arxiv2026,ih,axis1,no_signal,chat_roles,no_signal,first,"rule 7 (Amendment 2): chat_roles only when the method adds a role or turn, trains an order over roles or reads roles at inference; prompt wording and number-only labels are no_signal"
securecai_arxiv2026,second pass,defence set,(not coded),SecureCAI (full),(not coded),first,"kept as coded: full SecureCAI includes input sanitisation, which the extraction scope leaves out; the record codes SecureCAI without it"
spotlighting_arxiv2024,coverage,R2clean,proxy,direct,proxy,first,"kept: standard NLP benchmarks run through the defence's data transformation, a proxy for the user's task on clean data"
spotlighting_arxiv2024,spot_delimit_gpt35,isolation,ablation_present,no_ablation,ablation_present,first,rule 8: the report runs the method with a component removed or changed and reports the result
spotlighting_arxiv2024,spot_delimit_gpt35,signal_dependence,measured,baseline_only,measured,first,rule 9: the defence is run with its signal removed or moved
spotlighting_arxiv2024,second pass,defence set,(not coded),Defensive system instructions only,(not coded),first,"kept as coded: the instruction-only prompt is the signal-removed control of delimiting, recorded in spot_delimit_gpt35's signal_dependence_detail; its values are shown in a figure only"
tab_arxiv2026,coverage,R1,proxy,direct,direct,second,distractor execution is the rate at which the injected instruction is executed; R1 does not require the instruction to be malicious (p.24)
tab_arxiv2026,coverage,R3cp,not_reported,proxy,proxy,second,"task resolution needs the agent to act on the helpful cue placed in the file it reads, instruction-bearing data the task asks for (p.24)"
tab_arxiv2026,spot_mm,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tab_arxiv2026,spot_mm,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.24)"
tab_arxiv2026,spot_qw,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tab_arxiv2026,spot_qw,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.24)"
tab_arxiv2026,spot_g54,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tab_arxiv2026,spot_g54,signal_dependence,not_reported,baseline_only,baseline_only,second,"prompting or fixed inference rule with a source signal; its signal-removed condition is the undefended run, which is reported (p.24)"
tab_arxiv2026,rup_mm,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tab_arxiv2026,rup_qw,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tab_arxiv2026,rup_g54,isolation,not_applicable,no_ablation,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tracer_emnlp2026,coverage,R1,proxy,direct,proxy,first,"kept: the conditional ASR counts any wrong answer after the injection (p.3, equation 3), not the injected goal, so it is a proxy"
tracer_emnlp2026,coverage,R2attack,proxy,direct,direct,second,evidence accuracy scores the user's question under an injection; the report-level code disagreed with the coding of its own results (direct) (p.16)
tracer_emnlp2026,coverage,R3cp,not_reported,proxy,proxy,second,Table 6 scores answers that must use valid evidence carrying an embedded injection (p.16)
tracer_emnlp2026,rag3,signal_dependence,measured_other_checkpoint,measured,measured_other_checkpoint,first,rule 9: measured on another checkpoint of the method
tracer_emnlp2026,rag3,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,rag7,signal_dependence,not_reported,measured,measured_other_checkpoint,neither,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only (p.18)
tracer_emnlp2026,rag7,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,t6,isolation,not_applicable,ablation_present,ablation_present,second,the TRACER method itself on an unnamed checkpoint; its components are ablated (Tables 8 and 11) (p.8)
tracer_emnlp2026,t6,signal_dependence,not_reported,measured,measured_other_checkpoint,neither,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only (p.18)
tracer_emnlp2026,t6,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,t11,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tracer_emnlp2026,t11,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,t11,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,tool3,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,tool3,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,multi3,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,multi3,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,agent3,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,agent3,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,rag7e1,signal_dependence,not_reported,measured,measured_other_checkpoint,neither,Table 11 removes the RAG wrapper and the RAG system prompt on the 3B RAG specialist only (p.18)
tracer_emnlp2026,rag7e1,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,fu3,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tracer_emnlp2026,fu3,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,fu3,axis2,preference_optimisation;distillation,adversarial_rl;distillation,preference_optimisation;distillation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,abl_ans,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tracer_emnlp2026,abl_ans,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,abl_ans,axis2,preference_optimisation,adversarial_rl;distillation,preference_optimisation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop
tracer_emnlp2026,abl_comp,isolation,not_applicable,ablation_present,not_applicable,first,"rule 8 (Amendment 2): a variant or control record, or another report's defence not ablated here"
tracer_emnlp2026,abl_comp,signal_dependence,not_reported,measured,not_reported,first,rule 9: a trained defence with a signal and no run with the signal removed or moved
tracer_emnlp2026,abl_comp,axis2,preference_optimisation,adversarial_rl;distillation,preference_optimisation,first,kept: GRPO against fixed adversarial payloads; no adversary is trained in the loop

In [ ]:
dis = search.read_csv(paths.REPORTS / "second_pass_disagreements_04.csv")
res = search.read_csv(paths.REPORTS / "second_pass_resolution_04r.csv")
key = lambda r: (r["report"], r["unit"], r["field"], r["first"], r["second"])
if sorted(map(key, dis)) != sorted(map(key, res)):
    raise SystemExit("STOP: the resolution file does not list exactly the disagreements on file")
wrong = []
for r in res:
    if r["unit"] == "second pass":
        continue                                   # defence-set entries: no record field to read back
    unit = r["unit"] if r["unit"] in ("coverage", "report") else "defences:" + r["unit"]
    now = extract.encode(extract.field_value(recs[r["report"]], unit, r["field"]))
    if now != r["final"]:
        wrong.append(f"{r['report']}:{r['unit']}:{r['field']} final {r['final']!r}, on file {now!r}")
if wrong:
    raise SystemExit(f"STOP: resolution does not match the records: {wrong[:5]}")
tab = Counter((r["field"], r["resolved_to"]) for r in res)
print(f"{'field':<22} {'first':>6} {'second':>7} {'neither':>8}")
for f in sorted({r["field"] for r in res}, key=lambda f: -sum(v for (g, _), v in tab.items() if g == f)):
    print(f"{f:<22} {tab[(f, 'first')]:>6} {tab[(f, 'second')]:>7} {tab[(f, 'neither')]:>8}")
print(f"{len(res)} disagreements resolved; every final value read back from the re-coded records")

## Stage 6. Decision record and tests

In [ ]:
side = Counter(r["resolved_to"] for r in res)
seven = ("axis1", "isolation", "signal_dependence")
lines = [f"decision_04r: AMENDMENT 2 APPLIED; {len(recs)} RECORDS RE-CODED; AMENDED RULES HOLD IN EVERY RECORD",
         f"this step: {dt.date.today().isoformat()}",
         f"agreement figures before resolution: reports/agreement_04.csv, committed in {SP_COMMIT}",
         "rules amended: 7 (axis 1, chat roles and internal interventions), 8 (isolation, not_applicable defined), "
         "9 (signal dependence for prompting, fixed rules and trained defences), 32 (pooled results, R1, R3safe and R3cp proxies)",
         f"re-coded in every record: axis 1, isolation and signal dependence in {n_def} defence records; coverage R1, R3ns, R3cp and R3safe "
         f"in {len(recs)} reports (reports/recode_04r.csv)",
         f"changed: {len(changes)} fields in {written} record files; " + "; ".join(
             f"{f} {sum(1 for r in changes if r['field'] == f)}" for f in seven) +
         f"; coverage {sum(1 for r in changes if r['unit'] == 'coverage')}; results {sum(1 for r in changes if r['unit'].startswith('results:'))}; "
         f"other {sum(1 for r in changes if r['field'] not in seven and r['unit'] != 'coverage' and not r['unit'].startswith('results:'))}",
         f"amended-rule problems: {len(rules_before)} before, {len(rules_after)} after",
         f"second-pass disagreements resolved: {len(res)} (to the first code {side['first']}, to the second {side['second']}, "
         f"to neither {side['neither']}); reports/second_pass_resolution_04r.csv",
         "not changed: printed values, anchors, evaluations and comparisons",
         "next: notebook 04_full_extraction (one record per included study, CSV tables under reports/extraction/, decisive values)"]
(paths.REPORTS / "decision_04r.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 7. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04r rule amendment and re-code: coding rules 7, 8, 9 and 32 amended after the second pass; axis 1, isolation, "
                    "signal dependence and coverage of R1 and R3 re-checked in all 124 records; every second-pass disagreement resolved"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)